# 2025

In [8]:
import pandas as pd
import numpy as np
from pathlib import Path
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# 노트북 기준 상대 경로
RAW_2025   = Path("../data/1_raw/DATA_2025년 국민생활체육조사.xlsx")
OUTPUT_DIR = Path("../data/3_variable_define")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

YEAR = 2025
OUTPUT_FILE = OUTPUT_DIR / f"{YEAR}_variable_define.xlsx"
print("경로 설정 완료")

# ── 헤더 5행 로드
hdr = pd.read_excel(RAW_2025, header=None, nrows=5)

# row0: 섹션명 (병합셀 → ffill)
sec_row  = hdr.iloc[0].ffill()
# row1: 주 설명
desc1    = hdr.iloc[1]
# row2: 보조 설명 (지역→시도/시군구/읍면동 등)
desc2    = hdr.iloc[2]
# row4: 변수코드
code_row = hdr.iloc[4]

n_cols = len(code_row)
print(f"총 컬럼 수: {n_cols}")
print(f"섹션 목록: {sec_row.dropna().unique().tolist()}")

def build_description(i, desc1, desc2):
    """
    row1(주설명) + row2(보조설명) 조합
    - row1이 있으면 row1 사용
    - row1이 NaN이고 row2가 있으면 → 직전 row1 + ' | ' + row2
    - 둘 다 NaN이면 → 직전 row1 (순번 항목)
    """
    d1 = str(desc1.iloc[i]).strip() if not pd.isna(desc1.iloc[i]) else ""
    d2 = str(desc2.iloc[i]).strip() if not pd.isna(desc2.iloc[i]) else ""

    # 개행문자 정리
    d1 = d1.replace("\n", " ").strip()
    d2 = d2.replace("\n", " ").strip()

    if d1 and d2:
        return f"{d1} | {d2}"
    elif d1:
        return d1
    elif d2:
        # row1은 NaN → ffill로 부모 문항 찾기
        parent = ""
        for j in range(i - 1, -1, -1):
            pv = str(desc1.iloc[j]).strip() if not pd.isna(desc1.iloc[j]) else ""
            if pv:
                parent = pv.replace("\n", " ").strip()
                break
        return f"{parent} | {d2}" if parent else d2
    else:
        # 둘 다 NaN → 부모 문항명만
        for j in range(i - 1, -1, -1):
            pv = str(desc1.iloc[j]).strip() if not pd.isna(desc1.iloc[j]) else ""
            if pv:
                return pv.replace("\n", " ").strip()
        return ""

# 테스트 (처음 15개)
for i in range(15):
    code = code_row.iloc[i]
    sec  = sec_row.iloc[i]
    desc = build_description(i, desc1, desc2)
    print(f"  [{i:3d}] {str(code):<12}  섹션={str(sec)[:20]:<20}  설명={desc[:50]}")
    
    
    
# ── 실제 데이터 로드 (row4=변수코드, row5~=데이터)
df = pd.read_excel(RAW_2025, header=4)

# 첫 행이 변수코드 잔류 시 제거
if str(df.iloc[0, 0]).strip() == str(code_row.iloc[0]).strip():
    df = df.iloc[1:].reset_index(drop=True)

print(f"데이터: {df.shape[0]:,}행 × {df.shape[1]}열")
df.head(2)
MAX_UNIQ_DISPLAY = 50   # 유니크값 최대 표시 개수

rows = []
for i, code in enumerate(code_row):
    if pd.isna(code):
        continue
    code_str = str(code).strip()
    sec_str  = str(sec_row.iloc[i]).strip() if not pd.isna(sec_row.iloc[i]) else ""
    desc_str = build_description(i, desc1, desc2)

    # 실제 컬럼 존재 여부
    col_data = df[code_str] if code_str in df.columns else pd.Series(dtype=object)

    # 데이터 타입
    if pd.api.types.is_numeric_dtype(col_data):
        dtype = "숫자"
    else:
        dtype = "텍스트/혼합"

    # 유니크값
    uniq_vals = col_data.dropna().unique()
    try:
        uniq_sorted = sorted(uniq_vals, key=lambda x: (isinstance(x, str), x))
    except TypeError:
        uniq_sorted = list(uniq_vals)

    n_uniq = len(uniq_sorted)
    uniq_display = uniq_sorted[:MAX_UNIQ_DISPLAY]
    uniq_str = ", ".join(str(v) for v in uniq_display)
    if n_uniq > MAX_UNIQ_DISPLAY:
        uniq_str += "..."

    rows.append({
        "섹션":     sec_str,
        "변수코드": code_str,
        "변수설명": desc_str,
        "데이터타입": dtype,
        "유니크값수": n_uniq,
        "유니크값":  uniq_str,
        "표준컬럼명": "",  # 사용자 입력
        "제거여부":   "",  # Y 입력 시 제거
        "비고":       "",
    })

df_define = pd.DataFrame(rows)
print(f"변수정의서: {len(df_define)}행")
df_define.head(10)
# ── 섹션별 색상
SEC_COLORS = {
    "관리사항 및 응답자 정보": "D9E1F2",
    "Ⅰ. 건강과 체력상태":     "E2EFDA",
    "Ⅱ. 현재 체육활동 참여 현황": "FFF2CC",
    "Ⅲ. 체육활동 및 여건":    "FCE4D6",
    "Ⅴ. 개인 관련 사항":      "EDEDED",
}
DEFAULT_COLOR = "FFFFFF"

HDR_FILL = PatternFill("solid", fgColor="1F4E79")
HDR_FONT = Font(name="Arial", bold=True, color="FFFFFF", size=10)
BODY_FONT = Font(name="Arial", size=9)
thin = Side(style="thin", color="CCCCCC")
BORDER = Border(left=thin, right=thin, top=thin, bottom=thin)

# ── pandas로 먼저 저장
df_define.to_excel(OUTPUT_FILE, index=False)

# ── openpyxl로 서식 적용
wb = load_workbook(OUTPUT_FILE)
ws = wb.active
ws.title = f"{YEAR}_variable_define"
ws.freeze_panes = "A2"

# 헤더 서식
for cell in ws[1]:
    cell.fill = HDR_FILL
    cell.font = HDR_FONT
    cell.alignment = Alignment(horizontal="center", vertical="center")
    cell.border = BORDER
ws.row_dimensions[1].height = 20

# 데이터 서식
for r in range(2, ws.max_row + 1):
    sec_val = str(ws.cell(r, 1).value or "")
    fill_color = SEC_COLORS.get(sec_val, DEFAULT_COLOR)
    row_fill = PatternFill("solid", fgColor=fill_color)

    for c in range(1, ws.max_column + 1):
        cell = ws.cell(r, c)
        cell.fill = row_fill
        cell.font = BODY_FONT
        cell.alignment = Alignment(vertical="center", wrap_text=(c in (3, 6)))
        cell.border = BORDER
    ws.row_dimensions[r].height = 30

# 컬럼 너비
col_widths = [22, 14, 45, 12, 10, 60, 16, 10, 20]
for i, w in enumerate(col_widths, 1):
    ws.column_dimensions[get_column_letter(i)].width = w

# 안내 시트 추가
ws_guide = wb.create_sheet("안내")
guide_data = [
    ("컬럼명",    "설명"),
    ("섹션",      "변수 그룹 (색상 구분)"),
    ("변수코드",  "원본 엑셀의 컬럼명"),
    ("변수설명",  "원본 헤더에서 추출한 한글 설명"),
    ("데이터타입","숫자 / 텍스트·혼합"),
    ("유니크값수","해당 컬럼의 고유값 개수"),
    ("유니크값",  "실제 들어있는 값들 (최대 50개)"),
    ("표준컬럼명","★ 여기에 원하는 표준 변수명 입력 (비워두면 제거 대상)"),
    ("제거여부",  "★ Y 입력 시 제거 대상 (표준컬럼명이 있으면 무시됨)"),
    ("비고",      "메모"),
]
for row in guide_data:
    ws_guide.append(row)
ws_guide.column_dimensions["A"].width = 14
ws_guide.column_dimensions["B"].width = 50

wb.save(OUTPUT_FILE)
print(f"✅ 저장 완료: {OUTPUT_FILE.resolve()}")
print(f"   총 {len(df_define)}개 변수")
print(f"\n섹션별 변수 수:")
print(df_define["섹션"].value_counts().to_string())


경로 설정 완료
총 컬럼 수: 228
섹션 목록: ['관리사항 및 응답자 정보', 'Ⅰ. 건강과 체력상태', 'Ⅱ. 현재 체육활동 참여 현황', 'Ⅲ. 체육활동 및 여건', 'Ⅴ. 개인 관련 사항']
  [  0] ID            섹션=관리사항 및 응답자 정보         설명=ID
  [  1] 일련번호          섹션=관리사항 및 응답자 정보         설명=조사구 일련번호
  [  2] CODE1         섹션=관리사항 및 응답자 정보         설명=조사구번호
  [  3] CODE2         섹션=관리사항 및 응답자 정보         설명=가구번호
  [  4] APT           섹션=관리사항 및 응답자 정보         설명=주거유형
  [  5] CODE3         섹션=관리사항 및 응답자 정보         설명=지역 | 시도
  [  6] CODE4         섹션=관리사항 및 응답자 정보         설명=지역 | 시군구
  [  7] CODE5         섹션=관리사항 및 응답자 정보         설명=지역 | 읍면동
  [  8] CODE6         섹션=관리사항 및 응답자 정보         설명=읍면동(동부/읍면부)
  [  9] SEX           섹션=관리사항 및 응답자 정보         설명=성별
  [ 10] YEAR          섹션=관리사항 및 응답자 정보         설명=출생연도
  [ 11] MON           섹션=관리사항 및 응답자 정보         설명=출생월
  [ 12] AGE           섹션=관리사항 및 응답자 정보         설명=연령
  [ 13] Q1            섹션=Ⅰ. 건강과 체력상태           설명=문1. 본인의 건강상태에 대한 인식
  [ 14] Q2            섹션=Ⅰ. 건강과 체력상태           설명=문2. 건강 유지 위해 중요한 것
데이터: 9,000행 × 228열

In [9]:
from pathlib import Path
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ── 경로
RAW_2025     = Path("../data/1_raw/DATA_2025년 국민생활체육조사.xlsx")
VAR_DEF_2025 = Path("../data/3_variable_define/2025_variable_define.xlsx")
OUTPUT_DIR   = Path("../data/2_codebook")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE  = OUTPUT_DIR / "2025_codebook.xlsx"

# ── 데이터 로드
df_def = pd.read_excel(VAR_DEF_2025, sheet_name="2025_variable_define")
df_raw = pd.read_excel(RAW_2025, header=4)
if str(df_raw.iloc[0, 0]).strip() == "ID":
    df_raw = df_raw.iloc[1:].reset_index(drop=True)
print(f"변수정의서: {len(df_def)}개 / 원본데이터: {df_raw.shape[0]:,}행")

# ── 변수 유형 분류
SPORT_CODE_PATTERNS = (
    "Q6_", "Q8_1_1", "Q8_2_1", "Q8_3_1",
    "Q18_1", "Q18_2", "Q22_1", "Q22_2",
    "Q24_1_", "Q28_1_", "Q31_1_", "Q31_4_A",
)
MGMT_VARS       = {"ID", "일련번호", "CODE1", "CODE2", "APT"}
DEMO_CONTINUOUS = {"YEAR", "AGE", "MON"}
CONTINUOUS_KW   = ("비용", "소득", "기간", "일수", "시간", "분")

def classify_var(code, desc, dtype, n_uniq):
    if code.endswith(("t5","t6","t7","t8","t9","t10","t11","t12")) or "기타 응답" in desc:
        return "기타응답"
    if code in MGMT_VARS:
        return "관리변수"
    if code.startswith("CODE"):
        return "지역_텍스트" if dtype == "텍스트/혼합" else "관리변수"
    if code in DEMO_CONTINUOUS:
        return "인구통계_연속"
    if any(code.startswith(p) for p in SPORT_CODE_PATTERNS):
        return "종목·시설코드"
    if dtype == "텍스트/혼합":
        return "범주형_텍스트"
    if n_uniq == 0:
        return "응답없음"
    if n_uniq > 30 or any(kw in desc for kw in CONTINUOUS_KW):
        return "연속형"
    return "범주형_코드"

df_def["변수유형"] = df_def.apply(
    lambda r: classify_var(str(r["변수코드"]), str(r["변수설명"]),
                           str(r["데이터타입"]), int(r["유니크값수"])), axis=1
)
print("\n변수 유형 분포:")
print(df_def["변수유형"].value_counts().to_string())

# ── 코드북 Long Format 생성
MAX_VALUES = 300
rows = []

for _, var_row in df_def.iterrows():
    code   = str(var_row["변수코드"]).strip()
    desc   = str(var_row["변수설명"]).strip()
    sec    = str(var_row["섹션"]).strip()
    vtype  = str(var_row["변수유형"]).strip()
    n_uniq = int(var_row["유니크값수"])
    uniq_raw = str(var_row["유니크값"]).strip()
    base = dict(섹션=sec, 변수코드=code, 변수설명=desc, 변수유형=vtype)

    if vtype in ("연속형", "관리변수", "인구통계_연속", "응답없음"):
        rows.append({**base, "코드값": "(연속형/자유값)", "코드레이블": "",
                     "비고": f"유니크값 {n_uniq}개"})
        continue
    if vtype == "기타응답":
        rows.append({**base, "코드값": "(주관식)", "코드레이블": "", "비고": "텍스트 자유응답"})
        continue

    raw_vals = [v.strip() for v in uniq_raw.rstrip("...").split(",") if v.strip()] if uniq_raw not in ("nan","") else []
    if not raw_vals:
        rows.append({**base, "코드값": "(없음)", "코드레이블": "", "비고": ""})
        continue

    if vtype in ("범주형_텍스트", "지역_텍스트"):
        for val in raw_vals[:MAX_VALUES]:
            rows.append({**base, "코드값": val, "코드레이블": val,
                         "비고": f"전체 {n_uniq}개 중 {MAX_VALUES}개만 표시" if n_uniq > MAX_VALUES else ""})
        continue

    # 범주형_코드 / 종목·시설코드
    parsed = []
    for v in raw_vals:
        try:    parsed.append(int(float(v)))
        except: parsed.append(v)
    for val in sorted(set(parsed), key=lambda x: (isinstance(x, str), x)):
        rows.append({**base, "코드값": val, "코드레이블": "",
                     "비고": "종목/시설 분류코드" if vtype == "종목·시설코드" else ""})

df_codebook = pd.DataFrame(rows)
print(f"\n코드북 총 행수: {len(df_codebook):,}")

# ── Excel 저장
VTYPE_COLORS = {
    "범주형_코드":     "FFF2CC",
    "종목·시설코드":   "D9E1F2",
    "범주형_텍스트":   "E2EFDA",
    "지역_텍스트":     "E2EFDA",
    "연속형":          "EFEFEF",
    "인구통계_연속":   "EFEFEF",
    "관리변수":        "F3F3F3",
    "기타응답":        "FCE4D6",
    "응답없음":        "F4CCCC",
}
HDR_FILL  = PatternFill("solid", fgColor="1F4E79")
HDR_FONT  = Font(name="Arial", bold=True, color="FFFFFF", size=10)
BODY_FONT = Font(name="Arial", size=9)
INPUT_FONT = Font(name="Arial", size=9, color="1F4E79", bold=True)
thin   = Side(style="thin", color="CCCCCC")
BORDER = Border(left=thin, right=thin, top=thin, bottom=thin)

df_codebook.to_excel(OUTPUT_FILE, index=False, sheet_name="코드정의")
wb = load_workbook(OUTPUT_FILE)
ws = wb["코드정의"]
ws.freeze_panes = "A2"

for cell in ws[1]:
    cell.fill = HDR_FILL; cell.font = HDR_FONT
    cell.alignment = Alignment(horizontal="center", vertical="center")
    cell.border = BORDER
ws.row_dimensions[1].height = 22

# "코드레이블" 컬럼 인덱스 찾기
header_map = {cell.value: cell.column for cell in ws[1]}
col_label = header_map.get("코드레이블", 6)

for r in range(2, ws.max_row + 1):
    vtype = str(ws.cell(r, 4).value or "")
    row_fill = PatternFill("solid", fgColor=VTYPE_COLORS.get(vtype, "FFFFFF"))
    for c in range(1, ws.max_column + 1):
        cell = ws.cell(r, c)
        cell.fill = row_fill
        cell.font = INPUT_FONT if c == col_label and vtype in ("범주형_코드", "종목·시설코드") else BODY_FONT
        cell.alignment = Alignment(vertical="center", wrap_text=(c in (3,)))
        cell.border = BORDER
    ws.row_dimensions[r].height = 18

col_widths = [24, 14, 50, 16, 12, 28, 30]
for i, w in enumerate(col_widths, 1):
    if i <= ws.max_column:
        ws.column_dimensions[get_column_letter(i)].width = w

# 범례 시트
ws_leg = wb.create_sheet("범례_안내")
legend = [
    ("변수유형",        "설명",                         "처리 방법"),
    ("범주형_코드",     "숫자코드 → 한글레이블 필요",   "★ 코드레이블 컬럼에 직접 입력"),
    ("종목·시설코드",   "3자리 분류코드 (101, 401 등)", "★ 종목코드표 참조하여 입력"),
    ("범주형_텍스트",   "텍스트 값이 그대로 레이블",    "자동 입력 (수정 가능)"),
    ("지역_텍스트",     "시도·시군구·읍면동",           "자동 입력 (수정 가능)"),
    ("연속형",          "자유 수치값 (비용·소득 등)",   "입력 불필요"),
    ("인구통계_연속",   "출생연도·연령·출생월",         "입력 불필요"),
    ("관리변수",        "ID·조사구번호 등",             "입력 불필요"),
    ("기타응답",        "주관식 텍스트 응답",           "입력 불필요"),
    ("응답없음",        "해당 응답 없는 변수",          "제거 검토"),
]
leg_hdr_fill = PatternFill("solid", fgColor="1F4E79")
for i, row_data in enumerate(legend, 1):
    for j, val in enumerate(row_data, 1):
        c = ws_leg.cell(i, j, val)
        c.border = BORDER
        if i == 1:
            c.fill = leg_hdr_fill; c.font = HDR_FONT
            c.alignment = Alignment(horizontal="center", vertical="center")
        else:
            c.fill = PatternFill("solid", fgColor=VTYPE_COLORS.get(row_data[0], "FFFFFF"))
            c.font = BODY_FONT; c.alignment = Alignment(vertical="center")
    ws_leg.row_dimensions[i].height = 20
for i, w in enumerate([16, 30, 30], 1):
    ws_leg.column_dimensions[get_column_letter(i)].width = w

wb.save(OUTPUT_FILE)
print(f"\n✅ 저장 완료: {OUTPUT_FILE.resolve()}")

변수정의서: 228개 / 원본데이터: 9,000행

변수 유형 분포:
변수유형
범주형_코드     96
종목·시설코드    65
기타응답       26
연속형        20
응답없음        7
관리변수        5
지역_텍스트      4
인구통계_연속     3
범주형_텍스트     2

코드북 총 행수: 1,821

✅ 저장 완료: C:\Users\hanbi\OneDrive\Desktop\korean-sports-survey-ml\data\2_codebook\2025_codebook.xlsx


In [4]:
from pathlib import Path
import pandas as pd
import numpy as np

RAW_2025     = Path("../data/1_raw/DATA_2025년 국민생활체육조사.xlsx")
VAR_DEF_2025 = Path("../data/3_variable_define/2025_variable_define.xlsx")
CODEBOOK     = Path("../data/2_codebook/2025_codebook.xlsx")
OUTPUT_DIR   = Path("../data/5_processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. 원본 로드
df = pd.read_excel(RAW_2025, header=4)
if str(df.iloc[0, 0]).strip() == "ID":
    df = df.iloc[1:].reset_index(drop=True)
print(f"[원본] {df.shape[0]:,}행 × {df.shape[1]}열")

# 2. 변수정의서 → 제거 / 리네임
df_def = pd.read_excel(VAR_DEF_2025, sheet_name="2025_variable_define")
drop_cols = set(df_def.loc[
    df_def["제거여부"].astype(str).str.strip().str.upper() == "Y", "변수코드"
])
rename_map = {
    str(r["변수코드"]).strip(): str(r["표준컬럼명"]).strip()
    for _, r in df_def.iterrows()
    if str(r.get("표준컬럼명","")).strip() not in ("","nan")
    and str(r.get("표준컬럼명","")).strip() != str(r["변수코드"]).strip()
}
df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")
df = df.rename(columns=rename_map)
df = df.loc[:, ~df.columns.duplicated()]
print(f"제거: {len(drop_cols)}개 / 리네임: {len(rename_map)}개 → {df.shape[1]}열")

# 3. 코드북 → 한글 레이블 치환
df_cb = pd.read_excel(CODEBOOK, sheet_name="코드정의")
code_map = {}
for _, row in df_cb.iterrows():
    orig_col  = str(row["변수코드"]).strip()
    label_val = str(row.get("코드레이블","")).strip()
    code_val  = row["코드값"]
    if not label_val or label_val in ("nan",""):
        continue
    if str(code_val).strip() in ("(연속형/자유값)","(주관식)","(없음)","nan"):
        continue
    cur_col = rename_map.get(orig_col, orig_col)
    code_map.setdefault(cur_col, {})
    try:    code_map[cur_col][int(float(code_val))] = label_val
    except: pass
    code_map[cur_col][str(code_val).strip()] = label_val

for col, mapping in code_map.items():
    if col not in df.columns:
        continue
    if pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].astype(object)
    df[col] = df[col].map(lambda x: mapping.get(x, mapping.get(str(x).strip(), x)))
print(f"코드 치환: {len(code_map)}개 컬럼")

# 4. 연도 추가 & 저장
df.insert(0, "연도", 2025)
out_csv = OUTPUT_DIR / "survey_2025_standardized.csv"
df.to_csv(out_csv, index=False, encoding="utf-8-sig")
print(f"\n✅ [{df.shape[0]:,}행 × {df.shape[1]}열] 저장 완료: {out_csv}")

[원본] 9,000행 × 228열
제거: 0개 / 리네임: 0개 → 228열
코드 치환: 6개 컬럼

✅ [9,000행 × 229열] 저장 완료: ..\data\5_processed\survey_2025_standardized.csv


# 2024

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ── 경로
RAW_2024   = Path("../data/1_raw/DATA_2024년 국민생활체육조사.xlsx")
OUTPUT_DIR = Path("../data/3_variable_define")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = OUTPUT_DIR / "2024_variable_define.xlsx"

YEAR = 2024

# ══════════════════════════════════════════
# 1. 헤더 파싱
# 2024 구조: row0=변수설명, row1=변수코드, row2~=데이터
# ══════════════════════════════════════════
hdr = pd.read_excel(RAW_2024, header=None, nrows=2)
desc_row = hdr.iloc[0]   # 변수설명
code_row = hdr.iloc[1]   # 변수코드

# 실제 데이터 로드
df = pd.read_excel(RAW_2024, header=1)
if str(df.iloc[0, 0]).strip() == str(code_row.iloc[0]).strip():
    df = df.iloc[1:].reset_index(drop=True)

print(f"총 컬럼수: {len(code_row)}")
print(f"데이터: {df.shape[0]:,}행 × {df.shape[1]}열")

# ══════════════════════════════════════════
# 2. 섹션 자동 분류 (2024는 섹션행 없으므로 변수코드 패턴으로 분류)
# ══════════════════════════════════════════
def infer_section(code):
    code = str(code).strip()
    if code in ("ID","CODE1","CODE2","APT","CODE5","CODE6","LOC2",
                "SEX","YEAR","MON","AGE"):
        return "관리 및 응답자 정보"
    if code.startswith("Q1") or code in ("Q1","Q2","Q2_etc","Q3","Q4","Q4_etc") \
       or code.startswith("Q5"):
        return "Ⅰ. 건강과 체력상태"
    if code.startswith("Q6") or code.startswith("Q7") or code.startswith("Q8"):
        return "Ⅲ. 체육활동 및 여건"
    if code.startswith("Q9") or code.startswith("Q10") or code.startswith("Q11") \
       or code.startswith("Q12") or code.startswith("Q13") or code.startswith("Q14"):
        return "Ⅲ. 체육활동 및 여건"
    if code.startswith("Q15") or code.startswith("Q16") or code.startswith("Q17"):
        return "Ⅱ. 현재 체육활동 참여 현황"
    if code.startswith("Q18") or code.startswith("Q19") or code.startswith("Q20") \
       or code.startswith("Q21") or code.startswith("Q22") or code.startswith("Q23") \
       or code.startswith("Q24") or code.startswith("Q25") or code.startswith("Q26") \
       or code.startswith("Q27"):
        return "Ⅱ. 현재 체육활동 참여 현황"
    if code.startswith("Q28") or code.startswith("Q29") or code.startswith("Q30") \
       or code.startswith("Q31"):
        return "Ⅱ. 현재 체육활동 참여 현황"
    if code.startswith("DQ"):
        return "Ⅴ. 개인 관련 사항"
    if code.startswith("D_") or code == "WT":
        return "파생변수·가중치"
    return "기타"

# ══════════════════════════════════════════
# 3. 변수정의서 DataFrame 생성
# ══════════════════════════════════════════
MAX_UNIQ_DISPLAY = 50

rows = []
for i, code in enumerate(code_row):
    if pd.isna(code):
        continue
    code_str = str(code).strip()
    desc_str = str(desc_row.iloc[i]).strip().replace("\n", " ") \
               if not pd.isna(desc_row.iloc[i]) else ""
    sec_str  = infer_section(code_str)

    col_data = df[code_str] if code_str in df.columns else pd.Series(dtype=object)

    # 데이터 타입
    dtype = "숫자" if pd.api.types.is_numeric_dtype(col_data) else "텍스트/혼합"

    # 유니크값
    uniq_vals = col_data.dropna().unique()
    try:
        uniq_sorted = sorted(uniq_vals, key=lambda x: (isinstance(x, str), x))
    except TypeError:
        uniq_sorted = list(uniq_vals)

    n_uniq = len(uniq_sorted)
    uniq_str = ", ".join(str(v) for v in uniq_sorted[:MAX_UNIQ_DISPLAY])
    if n_uniq > MAX_UNIQ_DISPLAY:
        uniq_str += "..."

    rows.append({
        "섹션":       sec_str,
        "변수코드":   code_str,
        "변수설명":   desc_str,
        "데이터타입": dtype,
        "유니크값수": n_uniq,
        "유니크값":   uniq_str,
        "표준컬럼명": "",
        "제거여부":   "",
        "비고":       "",
    })

df_define = pd.DataFrame(rows)
print(f"\n변수정의서: {len(df_define)}개 변수")
print("\n섹션별 변수 수:")
print(df_define["섹션"].value_counts().to_string())

# ══════════════════════════════════════════
# 4. Excel 저장 (섹션별 색상)
# ══════════════════════════════════════════
SEC_COLORS = {
    "관리 및 응답자 정보":              "D9E1F2",
    "Ⅰ. 건강과 체력상태":              "E2EFDA",
    "Ⅱ. 현재 체육활동 참여 현황":      "FFF2CC",
    "Ⅲ. 체육활동 및 여건":             "FCE4D6",
    "Ⅴ. 개인 관련 사항":               "EDEDED",
    "파생변수·가중치":                  "F4CCCC",
}
HDR_FILL  = PatternFill("solid", fgColor="1F4E79")
HDR_FONT  = Font(name="Arial", bold=True, color="FFFFFF", size=10)
BODY_FONT = Font(name="Arial", size=9)
thin      = Side(style="thin", color="CCCCCC")
BORDER    = Border(left=thin, right=thin, top=thin, bottom=thin)

df_define.to_excel(OUTPUT_FILE, index=False)
wb = load_workbook(OUTPUT_FILE)
ws = wb.active
ws.title = f"{YEAR}_variable_define"
ws.freeze_panes = "A2"

for cell in ws[1]:
    cell.fill = HDR_FILL; cell.font = HDR_FONT
    cell.alignment = Alignment(horizontal="center", vertical="center")
    cell.border = BORDER
ws.row_dimensions[1].height = 20

for r in range(2, ws.max_row + 1):
    sec_val   = str(ws.cell(r, 1).value or "")
    row_fill  = PatternFill("solid", fgColor=SEC_COLORS.get(sec_val, "FFFFFF"))
    for c in range(1, ws.max_column + 1):
        cell = ws.cell(r, c)
        cell.fill = row_fill; cell.font = BODY_FONT
        cell.alignment = Alignment(vertical="center", wrap_text=(c in (3, 6)))
        cell.border = BORDER
    ws.row_dimensions[r].height = 30

col_widths = [24, 16, 50, 12, 10, 60, 16, 10, 20]
for i, w in enumerate(col_widths, 1):
    ws.column_dimensions[get_column_letter(i)].width = w

wb.save(OUTPUT_FILE)
print(f"\n✅ 저장 완료: {OUTPUT_FILE}")

총 컬럼수: 232
데이터: 9,000행 × 232열

변수정의서: 232개 변수

섹션별 변수 수:
섹션
Ⅰ. 건강과 체력상태         123
Ⅱ. 현재 체육활동 참여 현황     39
Ⅲ. 체육활동 및 여건         37
Ⅴ. 개인 관련 사항          12
관리 및 응답자 정보          11
파생변수·가중치             10

✅ 저장 완료: ..\data\3_variable_define\2024_variable_define.xlsx


In [14]:
from pathlib import Path
import pandas as pd
import numpy as np

YEAR      = 2024
MAPPING_F = Path(f"../data/4_mapping_table/{YEAR}_mapping.xlsx")
RAW_FILE  = Path(f"../data/1_raw/DATA_{YEAR}년 국민생활체육조사.xlsx")
CODEBOOK  = Path("../data/2_codebook/2025_codebook.xlsx")
OUT_DIR   = Path("../data/5_processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. 매핑 로드
df_var = pd.read_excel(MAPPING_F, sheet_name="변수_매핑", header=1)
df_val = pd.read_excel(MAPPING_F, sheet_name="값_매핑",   header=1)

rename_map, drop_cols = {}, set()
for _, row in df_var.iterrows():
    c_src = str(row.iloc[0]).strip()
    c_dst = str(row.iloc[3]).strip()
    if not c_src or c_src == "nan": continue
    if c_dst and c_dst != "nan": rename_map[c_src] = c_dst
    else: drop_cols.add(c_src)

val_rules = {}
for _, row in df_val.iterrows():
    유형  = str(row.iloc[0]).strip()
    c_src = str(row.iloc[2]).strip()
    c_dst = str(row.iloc[3]).strip()
    v_src = str(row.iloc[4]).strip()
    v_dst = str(row.iloc[5]).strip()
    if 유형 in ("분포차이(무시)","분포차이(무지)","보기 신규추가","동일"): continue
    if not c_src or c_src=="nan" or not c_dst or c_dst=="nan": continue
    if v_src in ("nan","(없음)","") or "..." in v_src or "(16개)" in v_src: continue
    cur_col = rename_map.get(c_src, c_src)
    val_rules.setdefault(cur_col, [])
    if 유형 == "보기 삭제됨":
        for code in [v.strip() for v in v_src.split(",") if v.strip()]:
            val_rules[cur_col].append((code, np.nan, "삭제"))
    elif 유형 in ("텍스트값 변경","숫자→텍스트","보기코드 변경"):
        val_rules[cur_col].append((v_src, v_dst, "치환"))

print(f"리네임: {len(rename_map)}개 / 제거: {len(drop_cols)}개 / 연도간 값변환: {len(val_rules)}컬럼")

# 2. 코드북 로드
df_cb = pd.read_excel(CODEBOOK, sheet_name="코드정의")
code_map = {}
for _, row in df_cb.iterrows():
    col       = str(row["변수코드"]).strip()
    label_val = str(row.get("코드레이블","")).strip()
    code_val  = row["코드값"]
    if not label_val or label_val in ("nan",""): continue
    if str(code_val).strip() in ("(연속형/자유값)","(주관식)","(없음)","nan"): continue
    code_map.setdefault(col, {})
    try:    code_map[col][int(float(code_val))] = label_val
    except: pass
    code_map[col][str(code_val).strip()] = label_val
print(f"코드북: {len(code_map)}개 컬럼")

# 3. 원본 로드
# 2024 구조: row0=한글설명, row1=변수코드, row2~=데이터
df = pd.read_excel(RAW_FILE, header=1)
print(f"원본: {df.shape[0]:,}행 × {df.shape[1]}열")
print(f"컬럼 확인 (앞 5개): {list(df.columns[:5])}")

# 4. 컬럼 제거 & 리네임
df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")
df = df.rename(columns=rename_map)
df = df.loc[:, ~df.columns.duplicated()]
print(f"리네임 후: {df.shape[1]}열")

# 5. 연도간 값 차이 처리
changed_diff = 0
for col, rules in val_rules.items():
    if col not in df.columns: continue
    has_text = any(isinstance(v,str) and v not in ("nan","") for _,v,_ in rules
                   if not (isinstance(v,float) and np.isnan(v)))
    if has_text and pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].astype(object)
    for v_src, v_dst, _ in rules:
        mask = df[col].astype(str).str.strip() == str(v_src).strip()
        try:
            nv = int(v_src) if str(v_src).isdigit() else float(v_src)
            mask = mask | (df[col] == nv)
        except: pass
        cnt = int(mask.sum())
        if cnt > 0:
            df.loc[mask, col] = v_dst
            changed_diff += cnt
print(f"연도간 값 변환: {changed_diff:,}건")

# 6. 코드북 레이블 치환
changed_label = 0
for col, mapping in code_map.items():
    if col not in df.columns: continue
    if pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].astype(object)
    before = df[col].copy()
    df[col] = df[col].map(lambda x: mapping.get(x, mapping.get(str(x).strip(), x)))
    changed_label += (before.astype(str) != df[col].astype(str)).sum()
print(f"코드북 레이블 치환: {changed_label:,}건")

# 7. 2025 컬럼 순서 맞춤
ref = Path("../data/5_processed/survey_2025_standardized.csv")
if ref.exists():
    cols_ref = [c for c in pd.read_csv(ref, nrows=0).columns if c != "연도"]
    ordered  = [c for c in cols_ref if c in df.columns]
    extra    = [c for c in df.columns if c not in cols_ref]
    df = df[ordered + extra]
    if extra: print(f"⚠️  {YEAR}에만 있는 컬럼 ({len(extra)}개): {extra}")

# 8. 연도 추가 & 저장
df.insert(0, "연도", YEAR)
out_csv = OUT_DIR / f"survey_{YEAR}_standardized.csv"
df.to_csv(out_csv, index=False, encoding="utf-8-sig")
print(f"\n✅ [{df.shape[0]:,}행 × {df.shape[1]}열] 저장 완료: {out_csv}")

리네임: 215개 / 제거: 17개 / 연도간 값변환: 19컬럼
코드북: 6개 컬럼
원본: 9,000행 × 232열
컬럼 확인 (앞 5개): ['ID', 'CODE1', 'CODE2', 'APT', 'CODE5']
리네임 후: 212열
연도간 값 변환: 9,463건
코드북 레이블 치환: 0건

✅ [9,000행 × 213열] 저장 완료: ..\data\5_processed\survey_2024_standardized.csv


# 2023

In [9]:
from pathlib import Path
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ── 경로
RAW_2023   = Path("../data/1_raw/DATA_2023년 국민생활체육조사.xlsx")
OUTPUT_DIR = Path("../data/3_variable_define")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = OUTPUT_DIR / "2023_variable_define.xlsx"

YEAR = 2023

# ══════════════════════════════════════════
# 1. 헤더 파싱
# 2023 구조: row0=빈행, row1=변수설명, row2=변수코드, row3~=데이터
# ══════════════════════════════════════════
hdr = pd.read_excel(RAW_2023, header=None, nrows=3)
desc_row = hdr.iloc[1]   # 변수설명
code_row = hdr.iloc[2]   # 변수코드

# 실제 데이터 로드
df = pd.read_excel(RAW_2023, header=2)
if str(df.iloc[0, 0]).strip() == str(code_row.iloc[0]).strip():
    df = df.iloc[1:].reset_index(drop=True)

print(f"총 컬럼수: {len(code_row)}")
print(f"데이터: {df.shape[0]:,}행 × {df.shape[1]}열")

# ══════════════════════════════════════════
# 2. 섹션 분류 (변수코드 패턴 기반)
# ══════════════════════════════════════════
def infer_section(code):
    code = str(code).strip()
    if code in ("ID","CODE1","CODE2","APT","CODE5","CODE6","LOC2","SEX","YEAR","MON","AGE"):
        return "관리 및 응답자 정보"
    if code in ("Q1","Q2","Q2_etc","Q3","Q4","Q4_etc") or code.startswith("Q5"):
        return "Ⅰ. 건강과 체력상태"
    if code.startswith("Q6") or code.startswith("Q7") or code.startswith("Q8"):
        return "Ⅲ. 체육활동 및 여건"
    if any(code.startswith(f"Q{n}") for n in range(9, 15)):
        return "Ⅲ. 체육활동 및 여건"
    if code.startswith("Q15") or code.startswith("Q16") or code.startswith("Q17"):
        return "Ⅱ. 현재 체육활동 참여 현황"
    if any(code.startswith(f"Q{n}") for n in range(18, 32)):
        return "Ⅱ. 현재 체육활동 참여 현황"
    if code.startswith("DQ"):
        return "Ⅴ. 개인 관련 사항"
    if code.startswith("D_") or code == "WT":
        return "파생변수·가중치"
    return "기타"

# ══════════════════════════════════════════
# 3. 변수정의서 DataFrame 생성
# ══════════════════════════════════════════
MAX_UNIQ_DISPLAY = 50
rows = []

for i, code in enumerate(code_row):
    if pd.isna(code):
        continue
    code_str = str(code).strip()
    desc_str = str(desc_row.iloc[i]).strip().replace("\n", " ") \
               if not pd.isna(desc_row.iloc[i]) else ""
    sec_str  = infer_section(code_str)

    col_data = df[code_str] if code_str in df.columns else pd.Series(dtype=object)
    dtype    = "숫자" if pd.api.types.is_numeric_dtype(col_data) else "텍스트/혼합"

    uniq_vals = col_data.dropna().unique()
    try:    uniq_sorted = sorted(uniq_vals, key=lambda x: (isinstance(x, str), x))
    except: uniq_sorted = list(uniq_vals)

    n_uniq   = len(uniq_sorted)
    uniq_str = ", ".join(str(v) for v in uniq_sorted[:MAX_UNIQ_DISPLAY])
    if n_uniq > MAX_UNIQ_DISPLAY:
        uniq_str += "..."

    rows.append({
        "섹션":       sec_str,
        "변수코드":   code_str,
        "변수설명":   desc_str,
        "데이터타입": dtype,
        "유니크값수": n_uniq,
        "유니크값":   uniq_str,
        "표준컬럼명": "",
        "제거여부":   "",
        "비고":       "",
    })

df_define = pd.DataFrame(rows)
print(f"\n변수정의서: {len(df_define)}개 변수")
print("\n섹션별 변수 수:")
print(df_define["섹션"].value_counts().to_string())

# ══════════════════════════════════════════
# 4. Excel 저장
# ══════════════════════════════════════════
SEC_COLORS = {
    "관리 및 응답자 정보":         "D9E1F2",
    "Ⅰ. 건강과 체력상태":         "E2EFDA",
    "Ⅱ. 현재 체육활동 참여 현황": "FFF2CC",
    "Ⅲ. 체육활동 및 여건":        "FCE4D6",
    "Ⅴ. 개인 관련 사항":          "EDEDED",
    "파생변수·가중치":             "F4CCCC",
}
HDR_FILL  = PatternFill("solid", fgColor="1F4E79")
HDR_FONT  = Font(name="Arial", bold=True, color="FFFFFF", size=10)
BODY_FONT = Font(name="Arial", size=9)
thin      = Side(style="thin", color="CCCCCC")
BORDER    = Border(left=thin, right=thin, top=thin, bottom=thin)

df_define.to_excel(OUTPUT_FILE, index=False)
wb = load_workbook(OUTPUT_FILE)
ws = wb.active
ws.title = f"{YEAR}_variable_define"
ws.freeze_panes = "A2"

for cell in ws[1]:
    cell.fill = HDR_FILL; cell.font = HDR_FONT
    cell.alignment = Alignment(horizontal="center", vertical="center")
    cell.border = BORDER
ws.row_dimensions[1].height = 20

for r in range(2, ws.max_row + 1):
    sec_val  = str(ws.cell(r, 1).value or "")
    row_fill = PatternFill("solid", fgColor=SEC_COLORS.get(sec_val, "FFFFFF"))
    for c in range(1, ws.max_column + 1):
        cell = ws.cell(r, c)
        cell.fill = row_fill; cell.font = BODY_FONT
        cell.alignment = Alignment(vertical="center", wrap_text=(c in (3, 6)))
        cell.border = BORDER
    ws.row_dimensions[r].height = 30

col_widths = [24, 16, 50, 12, 10, 60, 16, 10, 20]
for i, w in enumerate(col_widths, 1):
    ws.column_dimensions[get_column_letter(i)].width = w

wb.save(OUTPUT_FILE)
print(f"\n✅ 저장 완료: {OUTPUT_FILE}")

총 컬럼수: 231
데이터: 9,000행 × 231열

변수정의서: 231개 변수

섹션별 변수 수:
섹션
Ⅱ. 현재 체육활동 참여 현황    107
Ⅲ. 체육활동 및 여건         71
Ⅰ. 건강과 체력상태          12
기타                   11
Ⅴ. 개인 관련 사항          11
파생변수·가중치             10
관리 및 응답자 정보           9

✅ 저장 완료: ..\data\3_variable_define\2023_variable_define.xlsx


In [13]:
from pathlib import Path
import pandas as pd
import numpy as np

YEAR      = 2023
MAPPING_F = Path(f"../data/4_mapping_table/{YEAR}_mapping.xlsx")
RAW_FILE  = Path(f"../data/1_raw/DATA_{YEAR}년 국민생활체육조사.xlsx")
CODEBOOK  = Path("../data/2_codebook/2025_codebook.xlsx")
OUT_DIR   = Path("../data/5_processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. 매핑 로드
df_var = pd.read_excel(MAPPING_F, sheet_name="변수_매핑", header=1)
df_val = pd.read_excel(MAPPING_F, sheet_name="값_매핑",   header=1)

rename_map, drop_cols = {}, set()
for _, row in df_var.iterrows():
    c_src = str(row.iloc[0]).strip()
    c_dst = str(row.iloc[3]).strip()
    if not c_src or c_src == "nan": continue
    if c_dst and c_dst != "nan": rename_map[c_src] = c_dst
    else: drop_cols.add(c_src)

val_rules = {}
for _, row in df_val.iterrows():
    유형  = str(row.iloc[0]).strip()
    c_src = str(row.iloc[2]).strip()
    c_dst = str(row.iloc[3]).strip()
    v_src = str(row.iloc[4]).strip()
    v_dst = str(row.iloc[5]).strip()
    if 유형 in ("분포차이(무시)","분포차이(무지)","보기 신규추가","동일"): continue
    if not c_src or c_src=="nan" or not c_dst or c_dst=="nan": continue
    if v_src in ("nan","(없음)","") or "..." in v_src or "(16개)" in v_src: continue
    cur_col = rename_map.get(c_src, c_src)
    val_rules.setdefault(cur_col, [])
    if 유형 == "보기 삭제됨":
        for code in [v.strip() for v in v_src.split(",") if v.strip()]:
            val_rules[cur_col].append((code, np.nan, "삭제"))
    elif 유형 in ("텍스트값 변경","숫자→텍스트","보기코드 변경"):
        val_rules[cur_col].append((v_src, v_dst, "치환"))

print(f"리네임: {len(rename_map)}개 / 제거: {len(drop_cols)}개 / 연도간 값변환: {len(val_rules)}컬럼")

# 2. 코드북 로드
df_cb = pd.read_excel(CODEBOOK, sheet_name="코드정의")
code_map = {}
for _, row in df_cb.iterrows():
    col       = str(row["변수코드"]).strip()
    label_val = str(row.get("코드레이블","")).strip()
    code_val  = row["코드값"]
    if not label_val or label_val in ("nan",""): continue
    if str(code_val).strip() in ("(연속형/자유값)","(주관식)","(없음)","nan"): continue
    code_map.setdefault(col, {})
    try:    code_map[col][int(float(code_val))] = label_val
    except: pass
    code_map[col][str(code_val).strip()] = label_val
print(f"코드북: {len(code_map)}개 컬럼")

# 3. 원본 로드
# 2023 구조: row0=빈행, row1=한글설명, row2=변수코드, row3~=데이터
df = pd.read_excel(RAW_FILE, header=2)
print(f"원본: {df.shape[0]:,}행 × {df.shape[1]}열")
print(f"컬럼 확인 (앞 5개): {list(df.columns[:5])}")  # ID CODE1 APT... 확인용

# 4. 컬럼 제거 & 리네임
df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")
df = df.rename(columns=rename_map)
df = df.loc[:, ~df.columns.duplicated()]
print(f"리네임 후: {df.shape[1]}열")

# 5. 연도간 값 차이 처리
changed_diff = 0
for col, rules in val_rules.items():
    if col not in df.columns: continue
    has_text = any(isinstance(v,str) and v not in ("nan","") for _,v,_ in rules
                   if not (isinstance(v,float) and np.isnan(v)))
    if has_text and pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].astype(object)
    for v_src, v_dst, _ in rules:
        mask = df[col].astype(str).str.strip() == str(v_src).strip()
        try:
            nv = int(v_src) if str(v_src).isdigit() else float(v_src)
            mask = mask | (df[col] == nv)
        except: pass
        cnt = int(mask.sum())
        if cnt > 0:
            df.loc[mask, col] = v_dst
            changed_diff += cnt
print(f"연도간 값 변환: {changed_diff:,}건")

# 6. 코드북 레이블 치환
changed_label = 0
for col, mapping in code_map.items():
    if col not in df.columns: continue
    if pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].astype(object)
    before = df[col].copy()
    df[col] = df[col].map(lambda x: mapping.get(x, mapping.get(str(x).strip(), x)))
    changed_label += (before.astype(str) != df[col].astype(str)).sum()
print(f"코드북 레이블 치환: {changed_label:,}건")

# 7. 2025 컬럼 순서 맞춤
ref = Path("../data/5_processed/survey_2025_standardized.csv")
if ref.exists():
    cols_ref = [c for c in pd.read_csv(ref, nrows=0).columns if c != "연도"]
    ordered  = [c for c in cols_ref if c in df.columns]
    extra    = [c for c in df.columns if c not in cols_ref]
    df = df[ordered + extra]
    if extra: print(f"⚠️  {YEAR}에만 있는 컬럼 ({len(extra)}개): {extra}")

# 8. 연도 추가 & 저장
df.insert(0, "연도", YEAR)
out_csv = OUT_DIR / f"survey_{YEAR}_standardized.csv"
df.to_csv(out_csv, index=False, encoding="utf-8-sig")
print(f"\n✅ [{df.shape[0]:,}행 × {df.shape[1]}열] 저장 완료: {out_csv}")

리네임: 212개 / 제거: 19개 / 연도간 값변환: 16컬럼
코드북: 6개 컬럼
원본: 9,000행 × 231열
컬럼 확인 (앞 5개): ['ID', 'CODE1', 'APT', 'CODE5', 'CODE6']
리네임 후: 209열
연도간 값 변환: 0건
코드북 레이블 치환: 0건

✅ [9,000행 × 210열] 저장 완료: ..\data\5_processed\survey_2023_standardized.csv


# 2022

In [15]:
from pathlib import Path
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

YEAR        = 2022
RAW_2022    = Path(f"../data/1_raw/DATA_{YEAR}년 국민생활체육조사.xlsx")
OUTPUT_DIR  = Path("../data/3_variable_define")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = OUTPUT_DIR / f"{YEAR}_variable_define.xlsx"

# ══════════════════════════════════════════
# 1. 헤더 파싱
# 2022 구조: row0=설명, row1=변수코드, row2~=데이터
# ══════════════════════════════════════════
hdr      = pd.read_excel(RAW_2022, header=None, nrows=2)
desc_row = hdr.iloc[0]
code_row = hdr.iloc[1]

df = pd.read_excel(RAW_2022, header=1)
print(f"총 컬럼수: {len(code_row)}")
print(f"데이터: {df.shape[0]:,}행 × {df.shape[1]}열")

# ══════════════════════════════════════════
# 2. 섹션 분류 (2022는 A 접두사 체계)
# ══════════════════════════════════════════
def infer_section_2022(code):
    code = str(code).strip()
    # 관리정보
    if code in ("ID","AREA","CODE1","CODE2","CODE3","CODE4","CODE5","CODE6",
                "APT","SEX","YEAR","MON","AGE"):
        return "관리 및 응답자 정보"
    # 건강/체력
    if code.startswith("A0") and not code.startswith("A06") \
       or code in ("A01","A02","A03","A04","XA02","XA04") \
       or code.startswith("A05"):
        return "Ⅰ. 건강과 체력상태"
    if code in ("A06","A06A"):
        return "Ⅰ. 건강과 체력상태"
    # 체육활동 참여 현황 (A15~A31)
    if code.startswith("A15") or code.startswith("A16") or code.startswith("A17") \
       or code.startswith("A18") or code.startswith("A19") or code.startswith("A20") \
       or code.startswith("A21") or code.startswith("A22") or code.startswith("A23") \
       or code.startswith("A24") or code.startswith("A25") or code.startswith("A26") \
       or code.startswith("A27") or code.startswith("A28") or code.startswith("A29") \
       or code.startswith("A30") or code.startswith("A31") \
       or code.startswith("XA15") or code.startswith("XA17") or code.startswith("XA18") \
       or code.startswith("XA19") or code.startswith("XA20") or code.startswith("XA21") \
       or code.startswith("XA26") or code.startswith("XA27") or code.startswith("XA28") \
       or code.startswith("XA29") or code.startswith("XA30") or code.startswith("XA31"):
        return "Ⅱ. 현재 체육활동 참여 현황"
    # 체육활동 여건 (A07~A14)
    if code.startswith("A07") or code.startswith("A08") or code.startswith("A09") \
       or code.startswith("A10") or code.startswith("A11") or code.startswith("A12") \
       or code.startswith("A13") or code.startswith("A14") \
       or code.startswith("XA08") or code.startswith("XA10") or code.startswith("XA11") \
       or code.startswith("XA12") or code.startswith("XA13") or code.startswith("XA14"):
        return "Ⅲ. 체육활동 및 여건"
    # 개인사항
    if code.startswith("DQ"):
        return "Ⅴ. 개인 관련 사항"
    # 가중치
    if code == "WT":
        return "파생변수·가중치"
    return "기타"

# ══════════════════════════════════════════
# 3. 변수정의서 생성
# ══════════════════════════════════════════
MAX_UNIQ = 50
rows = []

for i, code in enumerate(code_row):
    if pd.isna(code):
        continue
    code_str = str(code).strip()
    desc_str = str(desc_row.iloc[i]).replace("\n"," ").strip() \
               if not pd.isna(desc_row.iloc[i]) else ""
    sec_str  = infer_section_2022(code_str)

    col_data = df[code_str] if code_str in df.columns else pd.Series(dtype=object)
    dtype    = "숫자" if pd.api.types.is_numeric_dtype(col_data) else "텍스트/혼합"

    uniq_vals = col_data.dropna().unique()
    try:    uniq_sorted = sorted(uniq_vals, key=lambda x: (isinstance(x, str), x))
    except: uniq_sorted = list(uniq_vals)

    n_uniq   = len(uniq_sorted)
    uniq_str = ", ".join(str(v) for v in uniq_sorted[:MAX_UNIQ])
    if n_uniq > MAX_UNIQ: uniq_str += "..."

    rows.append({
        "섹션":       sec_str,
        "변수코드":   code_str,
        "변수설명":   desc_str,
        "데이터타입": dtype,
        "유니크값수": n_uniq,
        "유니크값":   uniq_str,
        "표준컬럼명": "",
        "제거여부":   "",
        "비고":       "",
    })

df_define = pd.DataFrame(rows)
print(f"\n변수정의서: {len(df_define)}개 변수")
print("\n섹션별 변수 수:")
print(df_define["섹션"].value_counts().to_string())

# ══════════════════════════════════════════
# 4. Excel 저장
# ══════════════════════════════════════════
SEC_COLORS = {
    "관리 및 응답자 정보":         "D9E1F2",
    "Ⅰ. 건강과 체력상태":         "E2EFDA",
    "Ⅱ. 현재 체육활동 참여 현황": "FFF2CC",
    "Ⅲ. 체육활동 및 여건":        "FCE4D6",
    "Ⅴ. 개인 관련 사항":          "EDEDED",
    "파생변수·가중치":             "F4CCCC",
}
HDR_FILL  = PatternFill("solid", fgColor="1F4E79")
HDR_FONT  = Font(name="Arial", bold=True, color="FFFFFF", size=10)
BODY_FONT = Font(name="Arial", size=9)
thin      = Side(style="thin", color="CCCCCC")
BORDER    = Border(left=thin, right=thin, top=thin, bottom=thin)

df_define.to_excel(OUTPUT_FILE, index=False)
wb = load_workbook(OUTPUT_FILE)
ws = wb.active
ws.title = f"{YEAR}_variable_define"
ws.freeze_panes = "A2"

for cell in ws[1]:
    cell.fill = HDR_FILL; cell.font = HDR_FONT
    cell.alignment = Alignment(horizontal="center", vertical="center")
    cell.border = BORDER
ws.row_dimensions[1].height = 20

for r in range(2, ws.max_row + 1):
    sec_val  = str(ws.cell(r, 1).value or "")
    row_fill = PatternFill("solid", fgColor=SEC_COLORS.get(sec_val, "FFFFFF"))
    for c in range(1, ws.max_column + 1):
        cell = ws.cell(r, c)
        cell.fill = row_fill; cell.font = BODY_FONT
        cell.alignment = Alignment(vertical="center", wrap_text=(c in (3, 6)))
        cell.border = BORDER
    ws.row_dimensions[r].height = 30

col_widths = [24, 16, 50, 12, 10, 60, 16, 10, 20]
for i, w in enumerate(col_widths, 1):
    ws.column_dimensions[get_column_letter(i)].width = w

wb.save(OUTPUT_FILE)
print(f"\n✅ 저장 완료: {OUTPUT_FILE}")

총 컬럼수: 214
데이터: 9,000행 × 214열

변수정의서: 214개 변수

섹션별 변수 수:
섹션
Ⅱ. 현재 체육활동 참여 현황    112
Ⅲ. 체육활동 및 여건         44
Ⅰ. 건강과 체력상태          35
Ⅴ. 개인 관련 사항          11
관리 및 응답자 정보           9
기타                    2
파생변수·가중치              1

✅ 저장 완료: ..\data\3_variable_define\2022_variable_define.xlsx


In [16]:
from pathlib import Path
import pandas as pd
import numpy as np

YEAR      = 2022
MAPPING_F = Path(f"../data/4_mapping_table/{YEAR}_mapping.xlsx")
RAW_FILE  = Path(f"../data/1_raw/DATA_{YEAR}년 국민생활체육조사.xlsx")
CODEBOOK  = Path("../data/2_codebook/2025_codebook.xlsx")
OUT_DIR   = Path("../data/5_processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. 매핑 로드
df_var = pd.read_excel(MAPPING_F, sheet_name="변수_매핑", header=1)
df_val = pd.read_excel(MAPPING_F, sheet_name="값_매핑",   header=1)

rename_map, drop_cols = {}, set()
for _, row in df_var.iterrows():
    c_src = str(row.iloc[0]).strip()
    c_dst = str(row.iloc[3]).strip()
    if not c_src or c_src == "nan": continue
    if c_dst and c_dst != "nan": rename_map[c_src] = c_dst
    else: drop_cols.add(c_src)

val_rules = {}
for _, row in df_val.iterrows():
    유형  = str(row.iloc[0]).strip()
    c_src = str(row.iloc[2]).strip()
    c_dst = str(row.iloc[3]).strip()
    v_src = str(row.iloc[4]).strip()
    v_dst = str(row.iloc[5]).strip()
    if 유형 in ("분포차이(무시)","분포차이(무지)","보기 신규추가","동일"): continue
    if not c_src or c_src=="nan" or not c_dst or c_dst=="nan": continue
    if v_src in ("nan","(없음)","") or "..." in v_src: continue
    cur_col = rename_map.get(c_src, c_src)
    val_rules.setdefault(cur_col, [])
    if 유형 == "보기 삭제됨":
        for code in [v.strip() for v in v_src.split(",") if v.strip()]:
            val_rules[cur_col].append((code, np.nan, "삭제"))
    elif 유형 in ("텍스트값 변경","숫자→텍스트","보기코드 변경","숫자코드→시도명","텍스트 정리 (번호 제거)","텍스트 정리 + 슬래시 추가"):
        val_rules[cur_col].append((v_src, v_dst, "치환"))

print(f"리네임: {len(rename_map)}개 / 제거: {len(drop_cols)}개 / 값변환: {len(val_rules)}컬럼")

# 2. 코드북 로드
df_cb = pd.read_excel(CODEBOOK, sheet_name="코드정의")
code_map = {}
for _, row in df_cb.iterrows():
    col       = str(row["변수코드"]).strip()
    label_val = str(row.get("코드레이블","")).strip()
    code_val  = row["코드값"]
    if not label_val or label_val in ("nan",""): continue
    if str(code_val).strip() in ("(연속형/자유값)","(주관식)","(없음)","nan"): continue
    code_map.setdefault(col, {})
    try:    code_map[col][int(float(code_val))] = label_val
    except: pass
    code_map[col][str(code_val).strip()] = label_val
print(f"코드북: {len(code_map)}개 컬럼")

# 3. 원본 로드
# 2022 구조: row0=설명, row1=변수코드, row2~=데이터
df = pd.read_excel(RAW_FILE, header=1)
print(f"원본: {df.shape[0]:,}행 × {df.shape[1]}열")
print(f"컬럼 확인 (앞 5개): {list(df.columns[:5])}")  # ID AREA CODE1... 확인

# 4. 컬럼 제거 & 리네임
df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")
df = df.rename(columns=rename_map)
df = df.loc[:, ~df.columns.duplicated()]
print(f"리네임 후: {df.shape[1]}열")

# 5. 연도간 값 차이 처리 (AREA 숫자→시도명, CODE3→CODE6 텍스트 정리 등)
changed_diff = 0
for col, rules in val_rules.items():
    if col not in df.columns: continue
    has_text = any(isinstance(v,str) and v not in ("nan","") for _,v,_ in rules
                   if not (isinstance(v,float) and np.isnan(v)))
    if has_text and pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].astype(object)
    for v_src, v_dst, _ in rules:
        mask = df[col].astype(str).str.strip() == str(v_src).strip()
        try:
            nv = int(v_src) if str(v_src).isdigit() else float(v_src)
            mask = mask | (df[col] == nv)
        except: pass
        cnt = int(mask.sum())
        if cnt > 0:
            df.loc[mask, col] = v_dst
            changed_diff += cnt
print(f"연도간 값 변환: {changed_diff:,}건")

# 6. 코드북 레이블 치환
changed_label = 0
for col, mapping in code_map.items():
    if col not in df.columns: continue
    if pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].astype(object)
    before = df[col].copy()
    df[col] = df[col].map(lambda x: mapping.get(x, mapping.get(str(x).strip(), x)))
    changed_label += (before.astype(str) != df[col].astype(str)).sum()
print(f"코드북 레이블 치환: {changed_label:,}건")

# 7. 2025 컬럼 순서 맞춤
ref = Path("../data/5_processed/survey_2025_standardized.csv")
if ref.exists():
    cols_ref = [c for c in pd.read_csv(ref, nrows=0).columns if c != "연도"]
    ordered  = [c for c in cols_ref if c in df.columns]
    extra    = [c for c in df.columns if c not in cols_ref]
    df = df[ordered + extra]
    if extra: print(f"⚠️  {YEAR}에만 있는 컬럼 ({len(extra)}개): {extra}")

# 8. 연도 추가 & 저장
df.insert(0, "연도", YEAR)
out_csv = OUT_DIR / f"survey_{YEAR}_standardized.csv"
df.to_csv(out_csv, index=False, encoding="utf-8-sig")
print(f"\n✅ [{df.shape[0]:,}행 × {df.shape[1]}열] 저장 완료: {out_csv}")

리네임: 201개 / 제거: 13개 / 값변환: 18컬럼
코드북: 6개 컬럼
원본: 9,000행 × 214열
컬럼 확인 (앞 5개): ['ID', 'AREA', 'CODE1', 'CODE3', 'APT']
리네임 후: 201열
연도간 값 변환: 0건
코드북 레이블 치환: 0건

✅ [9,000행 × 202열] 저장 완료: ..\data\5_processed\survey_2022_standardized.csv


# 2021

In [17]:
from pathlib import Path
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

YEAR        = 2021
RAW_2021    = Path(f"../data/1_raw/DATA_{YEAR}년 국민생활체육조사.xlsx")
OUTPUT_DIR  = Path("../data/3_variable_define")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = OUTPUT_DIR / f"{YEAR}_variable_define.xlsx"

# ══════════════════════════════════════════
# 1. 헤더 파싱
# 2021 구조: row0=설명, row1=변수코드, row2~=데이터
# ══════════════════════════════════════════
hdr      = pd.read_excel(RAW_2021, header=None, nrows=2)
desc_row = hdr.iloc[0]
code_row = hdr.iloc[1]

df = pd.read_excel(RAW_2021, header=1)
print(f"총 컬럼수: {len(code_row)}")
print(f"데이터: {df.shape[0]:,}행 × {df.shape[1]}열")

# ══════════════════════════════════════════
# 2. 섹션 분류 (2021 혼합 체계)
# ══════════════════════════════════════════
def infer_section_2021(code):
    code = str(code).strip()
    # 관리정보
    if code in ("ID","CODE","LOC1","LOC2","HTYPE","SEX","YEAR","MON","AGE"):
        return "관리 및 응답자 정보"
    # 건강/체력 (Q01~Q06)
    if code in ("Q01","Q02","XQ02","Q03","Q04","XQ04") \
       or code.startswith("Q05") or code.startswith("XQ05") \
       or code in ("Q06","Q061"):
        return "Ⅰ. 건강과 체력상태"
    # 체육활동 참여 현황 (Q15~Q31)
    if code.startswith("Q15") or code.startswith("Q16") or code.startswith("Q17") \
       or code.startswith("Q18") or code.startswith("Q19") or code.startswith("Q20") \
       or code.startswith("Q21") or code.startswith("Q22") or code.startswith("Q23") \
       or code.startswith("Q24") or code.startswith("Q25") or code.startswith("Q26") \
       or code.startswith("Q27") or code.startswith("Q28") or code.startswith("Q29") \
       or code.startswith("Q30") or code.startswith("Q31") \
       or code.startswith("XQ15") or code.startswith("XQ17") or code.startswith("XQ18") \
       or code.startswith("XQ19") or code.startswith("XQ20") or code.startswith("XQ21") \
       or code.startswith("XQ26") or code.startswith("XQ27") or code.startswith("XQ28") \
       or code.startswith("XQ29") or code.startswith("XQ30") or code.startswith("XQ31"):
        return "Ⅱ. 현재 체육활동 참여 현황"
    # 체육활동 여건 (Q07~Q14)
    if code.startswith("Q07") or code.startswith("Q08") or code.startswith("Q09") \
       or code.startswith("Q10") or code.startswith("Q11") or code.startswith("Q12") \
       or code.startswith("Q13") or code.startswith("Q14") \
       or code.startswith("XQ08") or code.startswith("XQ10") or code.startswith("XQ11") \
       or code.startswith("XQ12") or code.startswith("XQ13") or code.startswith("XQ14"):
        return "Ⅲ. 체육활동 및 여건"
    # 개인사항 (EDU, MAR, FAM, CHI, INC, JOB, DQ)
    if code.startswith("EDU") or code.startswith("MAR") or code.startswith("FAM") \
       or code.startswith("CHI") or code.startswith("INC") or code.startswith("JOB") \
       or code.startswith("DQ"):
        return "Ⅴ. 개인 관련 사항"
    # 가중치
    if code == "WT":
        return "파생변수·가중치"
    return "기타"

# ══════════════════════════════════════════
# 3. 변수정의서 생성
# ══════════════════════════════════════════
MAX_UNIQ = 50
rows = []

for i, code in enumerate(code_row):
    if pd.isna(code):
        continue
    code_str = str(code).strip()
    desc_str = str(desc_row.iloc[i]).replace("\n"," ").strip() \
               if not pd.isna(desc_row.iloc[i]) else ""
    sec_str  = infer_section_2021(code_str)

    col_data = df[code_str] if code_str in df.columns else pd.Series(dtype=object)
    dtype    = "숫자" if pd.api.types.is_numeric_dtype(col_data) else "텍스트/혼합"

    uniq_vals = col_data.dropna().unique()
    try:    uniq_sorted = sorted(uniq_vals, key=lambda x: (isinstance(x, str), x))
    except: uniq_sorted = list(uniq_vals)

    n_uniq   = len(uniq_sorted)
    uniq_str = ", ".join(str(v) for v in uniq_sorted[:MAX_UNIQ])
    if n_uniq > MAX_UNIQ: uniq_str += "..."

    rows.append({
        "섹션":       sec_str,
        "변수코드":   code_str,
        "변수설명":   desc_str,
        "데이터타입": dtype,
        "유니크값수": n_uniq,
        "유니크값":   uniq_str,
        "표준컬럼명": "",
        "제거여부":   "",
        "비고":       "",
    })

df_define = pd.DataFrame(rows)
print(f"\n변수정의서: {len(df_define)}개 변수")
print("\n섹션별 변수 수:")
print(df_define["섹션"].value_counts().to_string())

# ══════════════════════════════════════════
# 4. Excel 저장
# ══════════════════════════════════════════
SEC_COLORS = {
    "관리 및 응답자 정보":         "D9E1F2",
    "Ⅰ. 건강과 체력상태":         "E2EFDA",
    "Ⅱ. 현재 체육활동 참여 현황": "FFF2CC",
    "Ⅲ. 체육활동 및 여건":        "FCE4D6",
    "Ⅴ. 개인 관련 사항":          "EDEDED",
    "파생변수·가중치":             "F4CCCC",
}
HDR_FILL  = PatternFill("solid", fgColor="1F4E79")
HDR_FONT  = Font(name="Arial", bold=True, color="FFFFFF", size=10)
BODY_FONT = Font(name="Arial", size=9)
thin      = Side(style="thin", color="CCCCCC")
BORDER    = Border(left=thin, right=thin, top=thin, bottom=thin)

df_define.to_excel(OUTPUT_FILE, index=False)
wb = load_workbook(OUTPUT_FILE)
ws = wb.active
ws.title = f"{YEAR}_variable_define"
ws.freeze_panes = "A2"

for cell in ws[1]:
    cell.fill = HDR_FILL; cell.font = HDR_FONT
    cell.alignment = Alignment(horizontal="center", vertical="center")
    cell.border = BORDER
ws.row_dimensions[1].height = 20

for r in range(2, ws.max_row + 1):
    sec_val  = str(ws.cell(r, 1).value or "")
    row_fill = PatternFill("solid", fgColor=SEC_COLORS.get(sec_val, "FFFFFF"))
    for c in range(1, ws.max_column + 1):
        cell = ws.cell(r, c)
        cell.fill = row_fill; cell.font = BODY_FONT
        cell.alignment = Alignment(vertical="center", wrap_text=(c in (3, 6)))
        cell.border = BORDER
    ws.row_dimensions[r].height = 30

col_widths = [24, 16, 50, 12, 10, 60, 16, 10, 20]
for i, w in enumerate(col_widths, 1):
    ws.column_dimensions[get_column_letter(i)].width = w

wb.save(OUTPUT_FILE)
print(f"\n✅ 저장 완료: {OUTPUT_FILE}")

총 컬럼수: 208
데이터: 9,000행 × 208열

변수정의서: 208개 변수

섹션별 변수 수:
섹션
Ⅱ. 현재 체육활동 참여 현황    112
Ⅲ. 체육활동 및 여건         60
Ⅰ. 건강과 체력상태          14
Ⅴ. 개인 관련 사항          11
관리 및 응답자 정보           9
기타                    1
파생변수·가중치              1

✅ 저장 완료: ..\data\3_variable_define\2021_variable_define.xlsx


In [18]:
from pathlib import Path
import pandas as pd
import numpy as np

YEAR      = 2021
MAPPING_F = Path(f"../data/4_mapping_table/{YEAR}_mapping.xlsx")
RAW_FILE  = Path(f"../data/1_raw/DATA_{YEAR}년 국민생활체육조사.xlsx")
CODEBOOK  = Path("../data/2_codebook/2025_codebook.xlsx")
OUT_DIR   = Path("../data/5_processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. 매핑 로드
df_var = pd.read_excel(MAPPING_F, sheet_name="변수_매핑", header=1)
df_val = pd.read_excel(MAPPING_F, sheet_name="값_매핑",   header=1)

rename_map, drop_cols = {}, set()
for _, row in df_var.iterrows():
    c_src = str(row.iloc[0]).strip()
    c_dst = str(row.iloc[3]).strip()
    if not c_src or c_src == "nan": continue
    if c_dst and c_dst != "nan": rename_map[c_src] = c_dst
    else: drop_cols.add(c_src)

val_rules = {}
for _, row in df_val.iterrows():
    유형  = str(row.iloc[0]).strip()
    c_src = str(row.iloc[2]).strip()
    c_dst = str(row.iloc[3]).strip()
    v_src = str(row.iloc[4]).strip()
    v_dst = str(row.iloc[5]).strip()
    if 유형 in ("분포차이(무시)","분포차이(무지)","보기 신규추가","동일"): continue
    if not c_src or c_src=="nan" or not c_dst or c_dst=="nan": continue
    if v_src in ("nan","(없음)","") or "..." in v_src: continue
    cur_col = rename_map.get(c_src, c_src)
    val_rules.setdefault(cur_col, [])
    if 유형 == "보기 삭제됨":
        for code in [v.strip() for v in v_src.split(",") if v.strip()]:
            val_rules[cur_col].append((code, np.nan, "삭제"))
    else:
        val_rules[cur_col].append((v_src, v_dst, "치환"))

print(f"리네임: {len(rename_map)}개 / 제거: {len(drop_cols)}개 / 값변환: {len(val_rules)}컬럼")

# 2. 코드북 로드
df_cb = pd.read_excel(CODEBOOK, sheet_name="코드정의")
code_map = {}
for _, row in df_cb.iterrows():
    col       = str(row["변수코드"]).strip()
    label_val = str(row.get("코드레이블","")).strip()
    code_val  = row["코드값"]
    if not label_val or label_val in ("nan",""): continue
    if str(code_val).strip() in ("(연속형/자유값)","(주관식)","(없음)","nan"): continue
    code_map.setdefault(col, {})
    try:    code_map[col][int(float(code_val))] = label_val
    except: pass
    code_map[col][str(code_val).strip()] = label_val
print(f"코드북: {len(code_map)}개 컬럼")

# 3. 원본 로드
# 2021 구조: row0=설명, row1=변수코드, row2~=데이터
df = pd.read_excel(RAW_FILE, header=1)
print(f"원본: {df.shape[0]:,}행 × {df.shape[1]}열")
print(f"컬럼 확인 (앞 5개): {list(df.columns[:5])}")  # ID CODE LOC1... 확인

# 4. 컬럼 제거 & 리네임
df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")
df = df.rename(columns=rename_map)
df = df.loc[:, ~df.columns.duplicated()]
print(f"리네임 후: {df.shape[1]}열")

# 5. 연도간 값 변환 (LOC1 숫자→시도명, LOC2 숫자→텍스트 등)
changed_diff = 0
for col, rules in val_rules.items():
    if col not in df.columns: continue
    has_text = any(isinstance(v,str) and v not in ("nan","") for _,v,_ in rules
                   if not (isinstance(v,float) and np.isnan(v)))
    if has_text and pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].astype(object)
    for v_src, v_dst, _ in rules:
        mask = df[col].astype(str).str.strip() == str(v_src).strip()
        try:
            nv = int(v_src) if str(v_src).isdigit() else float(v_src)
            mask = mask | (df[col] == nv)
        except: pass
        cnt = int(mask.sum())
        if cnt > 0:
            df.loc[mask, col] = v_dst
            changed_diff += cnt
print(f"연도간 값 변환: {changed_diff:,}건")

# 6. 코드북 레이블 치환
changed_label = 0
for col, mapping in code_map.items():
    if col not in df.columns: continue
    if pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].astype(object)
    before = df[col].copy()
    df[col] = df[col].map(lambda x: mapping.get(x, mapping.get(str(x).strip(), x)))
    changed_label += (before.astype(str) != df[col].astype(str)).sum()
print(f"코드북 레이블 치환: {changed_label:,}건")

# 7. 2025 컬럼 순서 맞춤
ref = Path("../data/5_processed/survey_2025_standardized.csv")
if ref.exists():
    cols_ref = [c for c in pd.read_csv(ref, nrows=0).columns if c != "연도"]
    ordered  = [c for c in cols_ref if c in df.columns]
    extra    = [c for c in df.columns if c not in cols_ref]
    df = df[ordered + extra]
    if extra: print(f"⚠️  {YEAR}에만 있는 컬럼 ({len(extra)}개): {extra}")

# 8. 연도 추가 & 저장
df.insert(0, "연도", YEAR)
out_csv = OUT_DIR / f"survey_{YEAR}_standardized.csv"
df.to_csv(out_csv, index=False, encoding="utf-8-sig")
print(f"\n✅ [{df.shape[0]:,}행 × {df.shape[1]}열] 저장 완료: {out_csv}")

리네임: 195개 / 제거: 13개 / 값변환: 17컬럼
코드북: 6개 컬럼
원본: 9,000행 × 208열
컬럼 확인 (앞 5개): ['ID', 'CODE', 'LOC1', 'LOC2', 'HTYPE']
리네임 후: 195열
연도간 값 변환: 0건
코드북 레이블 치환: 0건

✅ [9,000행 × 196열] 저장 완료: ..\data\5_processed\survey_2021_standardized.csv


# 2020

In [20]:
from pathlib import Path
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

YEAR        = 2020
RAW_2020    = Path(f"../data/1_raw/DATA_{YEAR}년 국민생활체육조사.xlsx")
OUTPUT_DIR  = Path("../data/3_variable_define")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = OUTPUT_DIR / f"{YEAR}_variable_define.xlsx"

# ══════════════════════════════════════════
# 1. 헤더 파싱
# 2020 구조: row0=설명, row1=변수코드, row2~=데이터
# ══════════════════════════════════════════
hdr      = pd.read_excel(RAW_2020, header=None, nrows=2)
desc_row = hdr.iloc[0]
code_row = hdr.iloc[1]

df = pd.read_excel(RAW_2020, header=1)
print(f"총 컬럼수: {len(code_row)}")
print(f"데이터: {df.shape[0]:,}행 × {df.shape[1]}열")

# ══════════════════════════════════════════
# 2. 섹션 분류
# ══════════════════════════════════════════
# 파생/관리변수 (제거 대상)
DROP_VARS = {"SQ5","SQ1_1","NSQ5","NSQ5_1","NSQ6","NAGE","NSEXAGE",
             "NDQ3_2","NDQ1","NDQ4","WT"}

def infer_section_2020(code):
    code = str(code).strip()
    if code in ("ID","SQ1","SQ1_1_TEXT","SQ3","SQ5_1_TEXT","SQ5_2_TEXT",
                "SQ5_3_TEXT","SQ6","SQ7","AGE"):
        return "관리 및 응답자 정보"
    if code in ("Q1","Q2","Q2_ETC","Q3","Q4","Q4_ETC") \
       or code.startswith("Q5") or code in ("Q6","Q6_1"):
        return "Ⅰ. 건강과 체력상태"
    if code.startswith("Q15") or code.startswith("Q16") or code.startswith("Q17") \
       or code.startswith("Q18") or code.startswith("Q19") or code.startswith("Q20") \
       or code.startswith("Q21") or code.startswith("Q22") or code.startswith("Q23") \
       or code.startswith("Q24") or code.startswith("Q25") or code.startswith("Q26") \
       or code.startswith("Q27") or code.startswith("Q28") or code.startswith("Q29") \
       or code.startswith("Q30") or code.startswith("Q31"):
        return "Ⅱ. 현재 체육활동 참여 현황"
    if code.startswith("Q7") or code.startswith("Q8") or code.startswith("Q9") \
       or code.startswith("Q10") or code.startswith("Q11") or code.startswith("Q12") \
       or code.startswith("Q13") or code.startswith("Q14"):
        return "Ⅲ. 체육활동 및 여건"
    if code.startswith("DQ"):
        return "Ⅴ. 개인 관련 사항"
    if code in DROP_VARS:
        return "파생변수·가중치"
    return "기타"

# ══════════════════════════════════════════
# 3. 변수정의서 생성
# ══════════════════════════════════════════
MAX_UNIQ = 50
rows = []

for i, code in enumerate(code_row):
    if pd.isna(code):
        continue
    code_str = str(code).strip()
    desc_str = str(desc_row.iloc[i]).replace("\n"," ").strip() \
               if not pd.isna(desc_row.iloc[i]) else ""
    sec_str  = infer_section_2020(code_str)

    col_data = df[code_str] if code_str in df.columns else pd.Series(dtype=object)
    dtype    = "숫자" if pd.api.types.is_numeric_dtype(col_data) else "텍스트/혼합"

    uniq_vals = col_data.dropna().unique()
    try:    uniq_sorted = sorted(uniq_vals, key=lambda x: (isinstance(x, str), x))
    except: uniq_sorted = list(uniq_vals)

    n_uniq   = len(uniq_sorted)
    uniq_str = ", ".join(str(v) for v in uniq_sorted[:MAX_UNIQ])
    if n_uniq > MAX_UNIQ: uniq_str += "..."

    # 제거여부 자동 표시
    제거 = "Y" if code_str in DROP_VARS else ""

    rows.append({
        "섹션":       sec_str,
        "변수코드":   code_str,
        "변수설명":   desc_str,
        "데이터타입": dtype,
        "유니크값수": n_uniq,
        "유니크값":   uniq_str,
        "표준컬럼명": "",
        "제거여부":   제거,
        "비고":       "",
    })

df_define = pd.DataFrame(rows)
print(f"\n변수정의서: {len(df_define)}개 변수")
print("\n섹션별 변수 수:")
print(df_define["섹션"].value_counts().to_string())

# ══════════════════════════════════════════
# 4. Excel 저장
# ══════════════════════════════════════════
SEC_COLORS = {
    "관리 및 응답자 정보":         "D9E1F2",
    "Ⅰ. 건강과 체력상태":         "E2EFDA",
    "Ⅱ. 현재 체육활동 참여 현황": "FFF2CC",
    "Ⅲ. 체육활동 및 여건":        "FCE4D6",
    "Ⅴ. 개인 관련 사항":          "EDEDED",
    "파생변수·가중치":             "F4CCCC",
}
HDR_FILL  = PatternFill("solid", fgColor="1F4E79")
HDR_FONT  = Font(name="Arial", bold=True, color="FFFFFF", size=10)
BODY_FONT = Font(name="Arial", size=9)
thin      = Side(style="thin", color="CCCCCC")
BORDER = Border(left=thin, right=thin, top=thin, bottom=thin)

df_define.to_excel(OUTPUT_FILE, index=False)
wb = load_workbook(OUTPUT_FILE)
ws = wb.active
ws.title = f"{YEAR}_variable_define"
ws.freeze_panes = "A2"

for cell in ws[1]:
    cell.fill = HDR_FILL; cell.font = HDR_FONT
    cell.alignment = Alignment(horizontal="center", vertical="center")
    cell.border = BORDER
ws.row_dimensions[1].height = 20

for r in range(2, ws.max_row + 1):
    sec_val  = str(ws.cell(r, 1).value or "")
    row_fill = PatternFill("solid", fgColor=SEC_COLORS.get(sec_val, "FFFFFF"))
    for c in range(1, ws.max_column + 1):
        cell = ws.cell(r, c)
        cell.fill = row_fill; cell.font = BODY_FONT
        cell.alignment = Alignment(vertical="center", wrap_text=(c in (3, 6)))
        cell.border = BORDER
    ws.row_dimensions[r].height = 30

col_widths = [24, 16, 50, 12, 10, 60, 16, 10, 20]
for i, w in enumerate(col_widths, 1):
    ws.column_dimensions[get_column_letter(i)].width = w

wb.save(OUTPUT_FILE)
print(f"\n✅ 저장 완료: {OUTPUT_FILE}")

총 컬럼수: 323
데이터: 9,000행 × 323열

변수정의서: 323개 변수

섹션별 변수 수:
섹션
Ⅱ. 현재 체육활동 참여 현황    157
Ⅲ. 체육활동 및 여건        119
Ⅰ. 건강과 체력상태          14
Ⅴ. 개인 관련 사항          12
파생변수·가중치             11
관리 및 응답자 정보          10

✅ 저장 완료: ..\data\3_variable_define\2020_variable_define.xlsx


In [21]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

YEAR      = 2020
MAPPING_F = Path(f"../data/4_mapping_table/{YEAR}_mapping.xlsx")
RAW_FILE  = Path(f"../data/1_raw/DATA_{YEAR}년 국민생활체육조사.xlsx")
CODEBOOK  = Path("../data/2_codebook/2025_codebook.xlsx")
OUT_DIR   = Path("../data/5_processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. 매핑 로드
df_var = pd.read_excel(MAPPING_F, sheet_name="변수_매핑", header=1)
df_val = pd.read_excel(MAPPING_F, sheet_name="값_매핑",   header=1)

rename_map, drop_cols = {}, set()
for _, row in df_var.iterrows():
    c_src = str(row.iloc[0]).strip()
    c_dst = str(row.iloc[3]).strip()
    if not c_src or c_src == "nan": continue
    if c_dst and c_dst != "nan": rename_map[c_src] = c_dst
    else: drop_cols.add(c_src)

# 값 변환 규칙 (SQ1_1_TEXT, SQ5_1_TEXT 단순 치환 제외 → 아래 별도 처리)
SKIP_COLS_VAL = {"SQ5_1_TEXT", "SQ1_1_TEXT"}
val_rules = {}
for _, row in df_val.iterrows():
    유형  = str(row.iloc[0]).strip()
    c_src = str(row.iloc[2]).strip()
    c_dst = str(row.iloc[3]).strip()
    v_src = str(row.iloc[4]).strip()
    v_dst = str(row.iloc[5]).strip()
    if c_src in SKIP_COLS_VAL: continue
    if 유형 in ("분포차이(무시)","분포차이(무지)","보기 신규추가","동일"): continue
    if not c_src or c_src=="nan" or not c_dst or c_dst=="nan": continue
    if v_src in ("nan","(없음)","") or "..." in v_src: continue
    cur_col = rename_map.get(c_src, c_src)
    val_rules.setdefault(cur_col, [])
    if 유형 == "보기 삭제됨":
        for code in [v.strip() for v in v_src.split(",") if v.strip()]:
            val_rules[cur_col].append((code, np.nan, "삭제"))
    else:
        val_rules[cur_col].append((v_src, v_dst, "치환"))

print(f"리네임: {len(rename_map)}개 / 제거: {len(drop_cols)}개 / 값변환: {len(val_rules)}컬럼")

# 2. 코드북 로드
df_cb = pd.read_excel(CODEBOOK, sheet_name="코드정의")
code_map = {}
for _, row in df_cb.iterrows():
    col       = str(row["변수코드"]).strip()
    label_val = str(row.get("코드레이블","")).strip()
    code_val  = row["코드값"]
    if not label_val or label_val in ("nan",""): continue
    if str(code_val).strip() in ("(연속형/자유값)","(주관식)","(없음)","nan"): continue
    code_map.setdefault(col, {})
    try:    code_map[col][int(float(code_val))] = label_val
    except: pass
    code_map[col][str(code_val).strip()] = label_val
print(f"코드북: {len(code_map)}개 컬럼")

# 3. 원본 로드
# 2020 구조: row0=설명, row1=변수코드, row2~=데이터
df = pd.read_excel(RAW_FILE, header=1)
print(f"원본: {df.shape[0]:,}행 × {df.shape[1]}열")

# ══════════════════════════════════════════
# 4. 2020 전용 전처리
# ══════════════════════════════════════════

# ── 4-1. SQ7 (YYYYMM) → YEAR + MON 분리
if "SQ7" in df.columns:
    sq7 = df["SQ7"].astype(str).str.strip().str.zfill(6)
    df["YEAR"] = pd.to_numeric(sq7.str[:4], errors="coerce")
    df["MON"]  = pd.to_numeric(sq7.str[4:6], errors="coerce")
    df = df.drop(columns=["SQ7"])
    print(f"SQ7 분리 완료 → YEAR, MON")

# ── 4-2. SQ5_1_TEXT 자유텍스트 → 표준 시도명
REGION_MAP = {
    r"서울":   "서울특별시",
    r"부산":   "부산광역시",
    r"대구":   "대구광역시",
    r"인천":   "인천광역시",
    r"광주":   "광주광역시",
    r"대전":   "대전광역시",
    r"울산":   "울산광역시",
    r"세종":   "세종특별자치시",
    r"경기":   "경기도",
    r"강원":   "강원특별자치도",
    r"충북|충청북":  "충청북도",
    r"충남|충청남":  "충청남도",
    r"전북|전라북":  "전라북도",
    r"전남|전라남":  "전라남도",
    r"경북|경상북":  "경상북도",
    r"경남|경상남":  "경상남도",
    r"제주":   "제주특별자치도",
}

def normalize_region(val):
    if pd.isna(val): return val
    s = str(val).strip()
    for pattern, std_name in REGION_MAP.items():
        if re.search(pattern, s):
            return std_name
    return val  # 매칭 실패 시 원본 유지

if "SQ5_1_TEXT" in df.columns:
    before = df["SQ5_1_TEXT"].copy()
    df["SQ5_1_TEXT"] = df["SQ5_1_TEXT"].apply(normalize_region)
    changed = (before.astype(str) != df["SQ5_1_TEXT"].astype(str)).sum()
    print(f"SQ5_1_TEXT 정규화: {changed:,}건")
    # 매칭 안 된 값 확인
    unmatched = df.loc[~df["SQ5_1_TEXT"].isin([v for v in REGION_MAP.values()]), "SQ5_1_TEXT"].unique()
    if len(unmatched): print(f"  ⚠️ 미매칭 값: {unmatched[:10]}")

# ── 4-3. SQ1_1_TEXT (동/읍/면) → CODE6 (동부/읍면부)
LOC_MAP = {"동": "동부", "읍": "읍/면부", "면": "읍/면부"}
if "SQ1_1_TEXT" in df.columns:
    df["SQ1_1_TEXT"] = df["SQ1_1_TEXT"].map(lambda x: LOC_MAP.get(str(x).strip(), x))
    print(f"SQ1_1_TEXT 정규화 완료")

# ── 4-4. Q17 축 변환 (속성_순위 → 순위_속성)
# 2020: Q17_속성_순위 / 2025: Q8_순위_속성
# rename_map에서 이미 처리됨 (Q17_1_1→Q8_1_1, Q17_1_2→Q8_2_1 등)
print("Q17 축 변환: rename_map 통해 처리")

# ══════════════════════════════════════════
# 5. 컬럼 제거 & 리네임
# ══════════════════════════════════════════
# SQ7은 이미 제거됨
drop_cols.discard("SQ7")
df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")
df = df.rename(columns=rename_map)
df = df.loc[:, ~df.columns.duplicated()]
print(f"리네임 후: {df.shape[1]}열")

# ══════════════════════════════════════════
# 6. 연도간 값 변환
# ══════════════════════════════════════════
changed_diff = 0
for col, rules in val_rules.items():
    if col not in df.columns: continue
    has_text = any(isinstance(v,str) and v not in ("nan","") for _,v,_ in rules
                   if not (isinstance(v,float) and np.isnan(v)))
    if has_text and pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].astype(object)
    for v_src, v_dst, _ in rules:
        mask = df[col].astype(str).str.strip() == str(v_src).strip()
        try:
            nv = int(v_src) if str(v_src).isdigit() else float(v_src)
            mask = mask | (df[col] == nv)
        except: pass
        cnt = int(mask.sum())
        if cnt > 0:
            df.loc[mask, col] = v_dst
            changed_diff += cnt
print(f"연도간 값 변환: {changed_diff:,}건")

# ══════════════════════════════════════════
# 7. 코드북 레이블 치환
# ══════════════════════════════════════════
changed_label = 0
for col, mapping in code_map.items():
    if col not in df.columns: continue
    if pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].astype(object)
    before = df[col].copy()
    df[col] = df[col].map(lambda x: mapping.get(x, mapping.get(str(x).strip(), x)))
    changed_label += (before.astype(str) != df[col].astype(str)).sum()
print(f"코드북 레이블 치환: {changed_label:,}건")

# ══════════════════════════════════════════
# 8. 2025 컬럼 순서 맞춤
# ══════════════════════════════════════════
ref = Path("../data/5_processed/survey_2025_standardized.csv")
if ref.exists():
    cols_ref = [c for c in pd.read_csv(ref, nrows=0).columns if c != "연도"]
    ordered  = [c for c in cols_ref if c in df.columns]
    extra    = [c for c in df.columns if c not in cols_ref]
    df = df[ordered + extra]
    if extra: print(f"⚠️  {YEAR}에만 있는 컬럼 ({len(extra)}개): {extra}")

# ══════════════════════════════════════════
# 9. 연도 추가 & 저장
# ══════════════════════════════════════════
df.insert(0, "연도", YEAR)
out_csv = OUT_DIR / f"survey_{YEAR}_standardized.csv"
df.to_csv(out_csv, index=False, encoding="utf-8-sig")
print(f"\n✅ [{df.shape[0]:,}행 × {df.shape[1]}열] 저장 완료: {out_csv}")

리네임: 194개 / 제거: 128개 / 값변환: 19컬럼
코드북: 6개 컬럼
원본: 9,000행 × 323열
SQ7 분리 완료 → YEAR, MON
SQ5_1_TEXT 정규화: 8,651건
  ⚠️ 미매칭 값: ['수원' '창원' '창원시' '청주' '전주' '포항' '포항시']
SQ1_1_TEXT 정규화 완료
Q17 축 변환: rename_map 통해 처리
리네임 후: 196열
연도간 값 변환: 0건
코드북 레이블 치환: 0건
⚠️  2020에만 있는 컬럼 (1개): ['SQ5_3_TEXT']

✅ [9,000행 × 197열] 저장 완료: ..\data\5_processed\survey_2020_standardized.csv


# 2019

In [23]:
from pathlib import Path
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

YEAR        = 2019
RAW_2019    = Path(f"../data/1_raw/DATA_{YEAR}년 국민생활체육조사.xlsx")
OUTPUT_DIR  = Path("../data/3_variable_define")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = OUTPUT_DIR / f"{YEAR}_variable_define.xlsx"

# 2019 구조: row0=설명, row1=변수코드, row2~=데이터
hdr      = pd.read_excel(RAW_2019, header=None, nrows=2)
desc_row = hdr.iloc[0]
code_row = hdr.iloc[1]

df = pd.read_excel(RAW_2019, header=1)
print(f"총 컬럼수: {len(code_row)}")
print(f"데이터: {df.shape[0]:,}행 × {df.shape[1]}열")

DROP_VARS = {"WT"}

def infer_section_2019(code):
    code = str(code).strip()
    if code in ("ID","조사구번호","시도","동읍면부","시군구","읍면동","HTYPE","SEX","BIRTH","AGE"):
        return "관리 및 응답자 정보"
    if code in ("Q01","Q02","Q02_ETC","Q03","Q04","Q04_ETC","Q06","Q061") \
       or code.startswith("Q05"):
        return "Ⅰ. 건강과 체력상태"
    if code.startswith("Q15") or code.startswith("Q16") or code.startswith("Q17") \
       or code.startswith("Q18") or code.startswith("Q19") or code.startswith("Q20") \
       or code.startswith("Q21") or code.startswith("Q22") or code.startswith("Q23") \
       or code.startswith("Q24") or code.startswith("Q25") or code.startswith("Q26") \
       or code.startswith("Q27") or code.startswith("Q28") or code.startswith("Q29") \
       or code.startswith("Q30") or code.startswith("Q31"):
        return "Ⅱ. 현재 체육활동 참여 현황"
    if code.startswith("Q07") or code.startswith("Q08") or code.startswith("Q09") \
       or code.startswith("Q10") or code.startswith("Q11") or code.startswith("Q12") \
       or code.startswith("Q13") or code.startswith("Q14"):
        return "Ⅲ. 체육활동 및 여건"
    if code.startswith("EDU") or code.startswith("MAR") or code.startswith("FAM") \
       or code.startswith("CHI") or code.startswith("INC") or code.startswith("JOB"):
        return "Ⅴ. 개인 관련 사항"
    if code in DROP_VARS:
        return "파생변수·가중치"
    return "기타"

MAX_UNIQ = 50
rows = []
for i, code in enumerate(code_row):
    if pd.isna(code): continue
    code_str = str(code).strip()
    desc_str = str(desc_row.iloc[i]).replace("\n"," ").strip() \
               if not pd.isna(desc_row.iloc[i]) else ""
    sec_str  = infer_section_2019(code_str)
    col_data = df[code_str] if code_str in df.columns else pd.Series(dtype=object)
    dtype    = "숫자" if pd.api.types.is_numeric_dtype(col_data) else "텍스트/혼합"
    uniq_vals = col_data.dropna().unique()
    try:    uniq_sorted = sorted(uniq_vals, key=lambda x: (isinstance(x, str), x))
    except: uniq_sorted = list(uniq_vals)
    n_uniq   = len(uniq_sorted)
    uniq_str = ", ".join(str(v) for v in uniq_sorted[:MAX_UNIQ])
    if n_uniq > MAX_UNIQ: uniq_str += "..."
    rows.append({
        "섹션":       sec_str,
        "변수코드":   code_str,
        "변수설명":   desc_str,
        "데이터타입": dtype,
        "유니크값수": n_uniq,
        "유니크값":   uniq_str,
        "표준컬럼명": "",
        "제거여부":   "Y" if code_str in DROP_VARS else "",
        "비고":       "",
    })

df_define = pd.DataFrame(rows)
print(f"\n변수정의서: {len(df_define)}개 변수")
print("\n섹션별 변수 수:")
print(df_define["섹션"].value_counts().to_string())

SEC_COLORS = {
    "관리 및 응답자 정보":         "D9E1F2",
    "Ⅰ. 건강과 체력상태":         "E2EFDA",
    "Ⅱ. 현재 체육활동 참여 현황": "FFF2CC",
    "Ⅲ. 체육활동 및 여건":        "FCE4D6",
    "Ⅴ. 개인 관련 사항":          "EDEDED",
    "파생변수·가중치":             "F4CCCC",
}
HDR_FILL  = PatternFill("solid", fgColor="1F4E79")
HDR_FONT  = Font(name="Arial", bold=True, color="FFFFFF", size=10)
BODY_FONT = Font(name="Arial", size=9)
thin      = Side(style="thin", color="CCCCCC")
BORDER    = Border(left=thin, right=thin, top=thin, bottom=thin)

df_define.to_excel(OUTPUT_FILE, index=False)
wb = load_workbook(OUTPUT_FILE)
ws = wb.active
ws.title = f"{YEAR}_variable_define"
ws.freeze_panes = "A2"

for cell in ws[1]:
    cell.fill = HDR_FILL; cell.font = HDR_FONT
    cell.alignment = Alignment(horizontal="center", vertical="center")
    cell.border = BORDER
ws.row_dimensions[1].height = 20

for r in range(2, ws.max_row + 1):
    sec_val  = str(ws.cell(r, 1).value or "")
    row_fill = PatternFill("solid", fgColor=SEC_COLORS.get(sec_val, "FFFFFF"))
    for c in range(1, ws.max_column + 1):
        cell = ws.cell(r, c)
        cell.fill = row_fill; cell.font = BODY_FONT
        cell.alignment = Alignment(vertical="center", wrap_text=(c in (3, 6)))
        cell.border = BORDER
    ws.row_dimensions[r].height = 30

col_widths = [24, 16, 50, 12, 10, 60, 16, 10, 20]
for i, w in enumerate(col_widths, 1):
    ws.column_dimensions[get_column_letter(i)].width = w

wb.save(OUTPUT_FILE)
print(f"\n✅ 저장 완료: {OUTPUT_FILE}")

총 컬럼수: 225
데이터: 9,000행 × 225열

변수정의서: 225개 변수

섹션별 변수 수:
섹션
Ⅱ. 현재 체육활동 참여 현황    118
Ⅲ. 체육활동 및 여건         70
Ⅰ. 건강과 체력상태          14
Ⅴ. 개인 관련 사항          11
관리 및 응답자 정보          10
기타                    1
파생변수·가중치              1

✅ 저장 완료: ..\data\3_variable_define\2019_variable_define.xlsx


In [24]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

YEAR      = 2019
MAPPING_F = Path(f"../data/4_mapping_table/{YEAR}_mapping.xlsx")
RAW_FILE  = Path(f"../data/1_raw/DATA_{YEAR}년 국민생활체육조사.xlsx")
CODEBOOK  = Path("../data/2_codebook/2025_codebook.xlsx")
OUT_DIR   = Path("../data/5_processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. 매핑 로드
df_var = pd.read_excel(MAPPING_F, sheet_name="변수_매핑", header=1)
df_val = pd.read_excel(MAPPING_F, sheet_name="값_매핑",   header=1)

# 시도/동읍면부는 별도 처리
SKIP_COLS_VAL = {"시도", "동읍면부"}

rename_map, drop_cols = {}, set()
for _, row in df_var.iterrows():
    c_src = str(row.iloc[0]).strip()
    c_dst = str(row.iloc[3]).strip()
    if not c_src or c_src == "nan": continue
    if c_dst and c_dst != "nan": rename_map[c_src] = c_dst
    else: drop_cols.add(c_src)

val_rules = {}
for _, row in df_val.iterrows():
    유형  = str(row.iloc[0]).strip()
    c_src = str(row.iloc[2]).strip()
    c_dst = str(row.iloc[3]).strip()
    v_src = str(row.iloc[4]).strip()
    v_dst = str(row.iloc[5]).strip()
    if c_src in SKIP_COLS_VAL: continue
    if 유형 in ("분포차이(무시)","분포차이(무지)","보기 신규추가","동일"): continue
    if not c_src or c_src=="nan" or not c_dst or c_dst=="nan": continue
    if v_src in ("nan","(없음)","") or "..." in v_src: continue
    cur_col = rename_map.get(c_src, c_src)
    val_rules.setdefault(cur_col, [])
    if 유형 == "보기 삭제됨":
        for code in [v.strip() for v in v_src.split(",") if v.strip()]:
            val_rules[cur_col].append((code, np.nan, "삭제"))
    else:
        val_rules[cur_col].append((v_src, v_dst, "치환"))

print(f"리네임: {len(rename_map)}개 / 제거: {len(drop_cols)}개 / 값변환: {len(val_rules)}컬럼")

# 2. 코드북 로드
df_cb = pd.read_excel(CODEBOOK, sheet_name="코드정의")
code_map = {}
for _, row in df_cb.iterrows():
    col       = str(row["변수코드"]).strip()
    label_val = str(row.get("코드레이블","")).strip()
    code_val  = row["코드값"]
    if not label_val or label_val in ("nan",""): continue
    if str(code_val).strip() in ("(연속형/자유값)","(주관식)","(없음)","nan"): continue
    code_map.setdefault(col, {})
    try:    code_map[col][int(float(code_val))] = label_val
    except: pass
    code_map[col][str(code_val).strip()] = label_val
print(f"코드북: {len(code_map)}개 컬럼")

# 3. 원본 로드
# 2019 구조: row0=설명, row1=변수코드, row2~=데이터
df = pd.read_excel(RAW_FILE, header=1)
print(f"원본: {df.shape[0]:,}행 × {df.shape[1]}열")

# ══════════════════════════════════════════
# 4. 2019 전용 전처리
# ══════════════════════════════════════════

# ── 4-1. BIRTH (YYYYMM) → YEAR + MON 분리
# BIRTH에 비정상값(4자리, 5자리 등) 존재 → 6자리만 정상 처리
if "BIRTH" in df.columns:
    birth_str = df["BIRTH"].astype(str).str.strip().str.zfill(6)
    # 6자리가 아닌 경우 NaN 처리
    valid_mask = birth_str.str.len() == 6
    df["YEAR"] = pd.to_numeric(birth_str.where(valid_mask).str[:4], errors="coerce")
    df["MON"]  = pd.to_numeric(birth_str.where(valid_mask).str[4:6], errors="coerce")
    df = df.drop(columns=["BIRTH"])
    invalid_cnt = (~valid_mask).sum()
    print(f"BIRTH 분리 완료 → YEAR, MON  (비정상값 {invalid_cnt}건 NaN 처리)")

# ── 4-2. 시도 → CODE3 (번호 접두사 제거)
# "01. 서울특별시" → "서울특별시", "10. 강원도" → "강원특별자치도"
SIDO_MAP = {
    "01": "서울특별시",  "02": "부산광역시",  "03": "대구광역시",
    "04": "인천광역시",  "05": "광주광역시",  "06": "대전광역시",
    "07": "울산광역시",  "08": "세종특별자치시", "09": "경기도",
    "10": "강원특별자치도", "11": "충청북도",  "12": "충청남도",
    "13": "전라북도",   "14": "전라남도",   "15": "경상북도",
    "16": "경상남도",   "17": "제주특별자치도",
}

def normalize_sido(val):
    if pd.isna(val): return val
    s = str(val).strip()
    # "01. 서울특별시" 형태 → 번호 추출
    m = re.match(r'^(\d{2})[.\s]', s)
    if m:
        code = m.group(1)
        if code in SIDO_MAP:
            return SIDO_MAP[code]
    return s  # 매칭 실패 시 원본

if "시도" in df.columns:
    before = df["시도"].copy()
    df["시도"] = df["시도"].apply(normalize_sido)
    changed = (before.astype(str) != df["시도"].astype(str)).sum()
    print(f"시도 정규화: {changed:,}건")
    unmatched = df.loc[~df["시도"].isin(SIDO_MAP.values()), "시도"].dropna().unique()
    if len(unmatched): print(f"  ⚠️ 미매칭: {unmatched[:5]}")

# ── 4-3. 동읍면부 → CODE6
LOC_MAP = {"1. 동부": "동부", "2. 읍면부": "읍/면부"}
if "동읍면부" in df.columns:
    df["동읍면부"] = df["동읍면부"].map(lambda x: LOC_MAP.get(str(x).strip(), x))
    print("동읍면부 정규화 완료")

# 5. 컬럼 제거 & 리네임
drop_cols.discard("BIRTH")  # 이미 처리됨
df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")
df = df.rename(columns=rename_map)
df = df.loc[:, ~df.columns.duplicated()]
print(f"리네임 후: {df.shape[1]}열")

# 6. 연도간 값 변환
changed_diff = 0
for col, rules in val_rules.items():
    if col not in df.columns: continue
    has_text = any(isinstance(v,str) and v not in ("nan","") for _,v,_ in rules
                   if not (isinstance(v,float) and np.isnan(v)))
    if has_text and pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].astype(object)
    for v_src, v_dst, _ in rules:
        mask = df[col].astype(str).str.strip() == str(v_src).strip()
        try:
            nv = int(v_src) if str(v_src).isdigit() else float(v_src)
            mask = mask | (df[col] == nv)
        except: pass
        cnt = int(mask.sum())
        if cnt > 0:
            df.loc[mask, col] = v_dst
            changed_diff += cnt
print(f"연도간 값 변환: {changed_diff:,}건")

# 7. 코드북 레이블 치환
changed_label = 0
for col, mapping in code_map.items():
    if col not in df.columns: continue
    if pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].astype(object)
    before = df[col].copy()
    df[col] = df[col].map(lambda x: mapping.get(x, mapping.get(str(x).strip(), x)))
    changed_label += (before.astype(str) != df[col].astype(str)).sum()
print(f"코드북 레이블 치환: {changed_label:,}건")

# 8. 2025 컬럼 순서 맞춤
ref = Path("../data/5_processed/survey_2025_standardized.csv")
if ref.exists():
    cols_ref = [c for c in pd.read_csv(ref, nrows=0).columns if c != "연도"]
    ordered  = [c for c in cols_ref if c in df.columns]
    extra    = [c for c in df.columns if c not in cols_ref]
    df = df[ordered + extra]
    if extra: print(f"⚠️  {YEAR}에만 있는 컬럼 ({len(extra)}개): {extra}")

# 9. 연도 추가 & 저장
df.insert(0, "연도", YEAR)
out_csv = OUT_DIR / f"survey_{YEAR}_standardized.csv"
df.to_csv(out_csv, index=False, encoding="utf-8-sig")
print(f"\n✅ [{df.shape[0]:,}행 × {df.shape[1]}열] 저장 완료: {out_csv}")

리네임: 190개 / 제거: 35개 / 값변환: 18컬럼
코드북: 6개 컬럼
원본: 9,000행 × 225열
BIRTH 분리 완료 → YEAR, MON  (비정상값 0건 NaN 처리)
시도 정규화: 9,000건
동읍면부 정규화 완료
리네임 후: 191열
연도간 값 변환: 0건
코드북 레이블 치환: 0건

✅ [9,000행 × 192열] 저장 완료: ..\data\5_processed\survey_2019_standardized.csv


# 2018

In [25]:
from pathlib import Path
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

YEAR        = 2018
RAW_2018    = Path(f"../data/1_raw/DATA_{YEAR}년 국민생활체육조사.xlsx")
OUTPUT_DIR  = Path("../data/3_variable_define")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = OUTPUT_DIR / f"{YEAR}_variable_define.xlsx"

# 2018 구조: row0=설명, row1=변수코드, row2~=데이터
hdr      = pd.read_excel(RAW_2018, header=None, nrows=2)
desc_row = hdr.iloc[0]
code_row = hdr.iloc[1]

df = pd.read_excel(RAW_2018, header=1)
print(f"총 컬럼수: {len(code_row)}")
print(f"데이터: {df.shape[0]:,}행 × {df.shape[1]}열")

# 파생변수/가중치 (제거 대상)
DROP_VARS = {"SEX","AGE","AREA","SCA","D_AREA","D_SCA","D_SEX","D_AGE",
             "D_SEXAGE","D_DQ1","D_DQ3","D_DQ4","WT"}

def infer_section_2018(code):
    code = str(code).strip()
    if code == "ID":
        return "관리 및 응답자 정보"
    if code in ("Q1","Q2","Q3","Q4","Q5_T") or code.startswith("Q5A") \
       or code in ("Q6","Q6A"):
        return "Ⅰ. 건강과 체력상태"
    if code.startswith("Q15") or code.startswith("Q16") or code.startswith("Q17") \
       or code.startswith("Q18") or code.startswith("Q19") or code.startswith("Q20") \
       or code.startswith("Q21") or code.startswith("Q22") or code.startswith("Q23") \
       or code.startswith("Q24") or code.startswith("Q25") or code.startswith("Q26") \
       or code.startswith("Q27") or code.startswith("Q28") or code.startswith("Q29") \
       or code.startswith("Q30"):
        return "Ⅱ. 현재 체육활동 참여 현황"
    if code.startswith("Q7") or code.startswith("Q8") or code.startswith("Q9") \
       or code.startswith("Q10") or code.startswith("Q11") or code.startswith("Q12") \
       or code.startswith("Q13") or code.startswith("Q14"):
        return "Ⅲ. 체육활동 및 여건"
    if code.startswith("DQ"):
        return "Ⅴ. 개인 관련 사항"
    if code in DROP_VARS:
        return "파생변수·가중치"
    return "기타"

MAX_UNIQ = 50
rows = []
for i, code in enumerate(code_row):
    if pd.isna(code): continue
    code_str = str(code).strip()
    desc_str = str(desc_row.iloc[i]).replace("\n"," ").strip() \
               if not pd.isna(desc_row.iloc[i]) else ""
    sec_str  = infer_section_2018(code_str)
    col_data = df[code_str] if code_str in df.columns else pd.Series(dtype=object)
    dtype    = "숫자" if pd.api.types.is_numeric_dtype(col_data) else "텍스트/혼합"
    uniq_vals = col_data.dropna().unique()
    try:    uniq_sorted = sorted(uniq_vals, key=lambda x: (isinstance(x, str), x))
    except: uniq_sorted = list(uniq_vals)
    n_uniq   = len(uniq_sorted)
    uniq_str = ", ".join(str(v) for v in uniq_sorted[:MAX_UNIQ])
    if n_uniq > MAX_UNIQ: uniq_str += "..."
    rows.append({
        "섹션":       sec_str,
        "변수코드":   code_str,
        "변수설명":   desc_str,
        "데이터타입": dtype,
        "유니크값수": n_uniq,
        "유니크값":   uniq_str,
        "표준컬럼명": "",
        "제거여부":   "Y" if code_str in DROP_VARS else "",
        "비고":       "",
    })

df_define = pd.DataFrame(rows)
print(f"\n변수정의서: {len(df_define)}개 변수")
print("\n섹션별 변수 수:")
print(df_define["섹션"].value_counts().to_string())

SEC_COLORS = {
    "관리 및 응답자 정보":         "D9E1F2",
    "Ⅰ. 건강과 체력상태":         "E2EFDA",
    "Ⅱ. 현재 체육활동 참여 현황": "FFF2CC",
    "Ⅲ. 체육활동 및 여건":        "FCE4D6",
    "Ⅴ. 개인 관련 사항":          "EDEDED",
    "파생변수·가중치":             "F4CCCC",
}
HDR_FILL  = PatternFill("solid", fgColor="1F4E79")
HDR_FONT  = Font(name="Arial", bold=True, color="FFFFFF", size=10)
BODY_FONT = Font(name="Arial", size=9)
thin      = Side(style="thin", color="CCCCCC")
BORDER    = Border(left=thin, right=thin, top=thin, bottom=thin)

df_define.to_excel(OUTPUT_FILE, index=False)
wb = load_workbook(OUTPUT_FILE)
ws = wb.active
ws.title = f"{YEAR}_variable_define"
ws.freeze_panes = "A2"

for cell in ws[1]:
    cell.fill = HDR_FILL; cell.font = HDR_FONT
    cell.alignment = Alignment(horizontal="center", vertical="center")
    cell.border = BORDER
ws.row_dimensions[1].height = 20

for r in range(2, ws.max_row + 1):
    sec_val  = str(ws.cell(r, 1).value or "")
    row_fill = PatternFill("solid", fgColor=SEC_COLORS.get(sec_val, "FFFFFF"))
    for c in range(1, ws.max_column + 1):
        cell = ws.cell(r, c)
        cell.fill = row_fill; cell.font = BODY_FONT
        cell.alignment = Alignment(vertical="center", wrap_text=(c in (3, 6)))
        cell.border = BORDER
    ws.row_dimensions[r].height = 30

col_widths = [24, 16, 50, 12, 10, 60, 16, 10, 20]
for i, w in enumerate(col_widths, 1):
    ws.column_dimensions[get_column_letter(i)].width = w

wb.save(OUTPUT_FILE)
print(f"\n✅ 저장 완료: {OUTPUT_FILE}")

총 컬럼수: 434
데이터: 9,000행 × 434열

변수정의서: 434개 변수

섹션별 변수 수:
섹션
Ⅱ. 현재 체육활동 참여 현황    305
Ⅲ. 체육활동 및 여건         92
파생변수·가중치             13
Ⅰ. 건강과 체력상태          12
Ⅴ. 개인 관련 사항          11
관리 및 응답자 정보           1

✅ 저장 완료: ..\data\3_variable_define\2018_variable_define.xlsx


In [26]:
from pathlib import Path
import pandas as pd
import numpy as np

YEAR      = 2018
MAPPING_F = Path(f"../data/4_mapping_table/{YEAR}_mapping.xlsx")
RAW_FILE  = Path(f"../data/1_raw/DATA_{YEAR}년 국민생활체육조사.xlsx")
CODEBOOK  = Path("../data/2_codebook/2025_codebook.xlsx")
OUT_DIR   = Path("../data/5_processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. 매핑 로드
df_var = pd.read_excel(MAPPING_F, sheet_name="변수_매핑", header=1)
df_val = pd.read_excel(MAPPING_F, sheet_name="값_매핑",   header=1)

rename_map, drop_cols = {}, set()
for _, row in df_var.iterrows():
    c_src = str(row.iloc[0]).strip()
    c_dst = str(row.iloc[3]).strip()
    if not c_src or c_src == "nan": continue
    if c_dst and c_dst != "nan": rename_map[c_src] = c_dst
    else: drop_cols.add(c_src)

val_rules = {}
for _, row in df_val.iterrows():
    유형  = str(row.iloc[0]).strip()
    c_src = str(row.iloc[2]).strip()
    c_dst = str(row.iloc[3]).strip()
    v_src = str(row.iloc[4]).strip()
    v_dst = str(row.iloc[5]).strip()
    if 유형 in ("분포차이(무시)","분포차이(무지)","보기 신규추가","동일"): continue
    if not c_src or c_src=="nan" or not c_dst or c_dst=="nan": continue
    if v_src in ("nan","(없음)","") or "..." in v_src: continue
    cur_col = rename_map.get(c_src, c_src)
    val_rules.setdefault(cur_col, [])
    if 유형 == "보기 삭제됨":
        for code in [v.strip() for v in v_src.split(",") if v.strip()]:
            val_rules[cur_col].append((code, np.nan, "삭제"))
    else:
        val_rules[cur_col].append((v_src, v_dst, "치환"))

print(f"리네임: {len(rename_map)}개 / 제거: {len(drop_cols)}개 / 값변환: {len(val_rules)}컬럼")

# 2. 코드북 로드
df_cb = pd.read_excel(CODEBOOK, sheet_name="코드정의")
code_map = {}
for _, row in df_cb.iterrows():
    col       = str(row["변수코드"]).strip()
    label_val = str(row.get("코드레이블","")).strip()
    code_val  = row["코드값"]
    if not label_val or label_val in ("nan",""): continue
    if str(code_val).strip() in ("(연속형/자유값)","(주관식)","(없음)","nan"): continue
    code_map.setdefault(col, {})
    try:    code_map[col][int(float(code_val))] = label_val
    except: pass
    code_map[col][str(code_val).strip()] = label_val
print(f"코드북: {len(code_map)}개 컬럼")

# 3. 원본 로드
# 2018 구조: row0=설명, row1=변수코드, row2~=데이터
df = pd.read_excel(RAW_FILE, header=1)
print(f"원본: {df.shape[0]:,}행 × {df.shape[1]}열")

# 4. 컬럼 제거 & 리네임
df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")
df = df.rename(columns=rename_map)
df = df.loc[:, ~df.columns.duplicated()]
print(f"리네임 후: {df.shape[1]}열")

# 5. 연도간 값 변환 (AREA 숫자→시도명)
changed_diff = 0
for col, rules in val_rules.items():
    if col not in df.columns: continue
    has_text = any(isinstance(v,str) and v not in ("nan","") for _,v,_ in rules
                   if not (isinstance(v,float) and np.isnan(v)))
    if has_text and pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].astype(object)
    for v_src, v_dst, _ in rules:
        mask = df[col].astype(str).str.strip() == str(v_src).strip()
        try:
            nv = int(v_src) if str(v_src).isdigit() else float(v_src)
            mask = mask | (df[col] == nv)
        except: pass
        cnt = int(mask.sum())
        if cnt > 0:
            df.loc[mask, col] = v_dst
            changed_diff += cnt
print(f"연도간 값 변환: {changed_diff:,}건")

# 6. 코드북 레이블 치환
changed_label = 0
for col, mapping in code_map.items():
    if col not in df.columns: continue
    if pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].astype(object)
    before = df[col].copy()
    df[col] = df[col].map(lambda x: mapping.get(x, mapping.get(str(x).strip(), x)))
    changed_label += (before.astype(str) != df[col].astype(str)).sum()
print(f"코드북 레이블 치환: {changed_label:,}건")

# 7. 2025 컬럼 순서 맞춤
ref = Path("../data/5_processed/survey_2025_standardized.csv")
if ref.exists():
    cols_ref = [c for c in pd.read_csv(ref, nrows=0).columns if c != "연도"]
    ordered  = [c for c in cols_ref if c in df.columns]
    extra    = [c for c in df.columns if c not in cols_ref]
    df = df[ordered + extra]
    if extra: print(f"⚠️  {YEAR}에만 있는 컬럼 ({len(extra)}개): {extra}")

# 8. 연도 추가 & 저장
df.insert(0, "연도", YEAR)
out_csv = OUT_DIR / f"survey_{YEAR}_standardized.csv"
df.to_csv(out_csv, index=False, encoding="utf-8-sig")
print(f"\n✅ [{df.shape[0]:,}행 × {df.shape[1]}열] 저장 완료: {out_csv}")

리네임: 153개 / 제거: 281개 / 값변환: 16컬럼
코드북: 6개 컬럼
원본: 9,000행 × 434열
리네임 후: 153열
연도간 값 변환: 0건
코드북 레이블 치환: 0건

✅ [9,000행 × 154열] 저장 완료: ..\data\5_processed\survey_2018_standardized.csv


# 2017

In [27]:
from pathlib import Path
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

YEAR        = 2017
RAW_2017    = Path(f"../data/1_raw/DATA_{YEAR}년 국민생활체육조사.xlsx")
OUTPUT_DIR  = Path("../data/3_variable_define")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = OUTPUT_DIR / f"{YEAR}_variable_define.xlsx"

# ══════════════════════════════════════════
# 2017 구조: row0=변수코드, row1~=데이터 (설명행 없음!)
# ══════════════════════════════════════════
code_row = pd.read_excel(RAW_2017, header=None, nrows=1).iloc[0]
df = pd.read_excel(RAW_2017, header=0)   # header=0 → 첫 행이 컬럼명

print(f"총 컬럼수: {len(code_row)}")
print(f"데이터: {df.shape[0]:,}행 × {df.shape[1]}열")

DROP_VARS = {"WT"}

def infer_section_2017(code):
    code = str(code).strip()
    if code in ("ID","CITY","AREA","GENDER","AGE1","AGE2","SUR"):
        return "관리 및 응답자 정보"
    if code in ("Q1","Q2","Q3","Q4","Q6","Q6A") or code.startswith("Q5"):
        return "Ⅰ. 건강과 체력상태"
    if code.startswith("Q15") or code.startswith("Q16") or code.startswith("Q17") \
       or code.startswith("Q18") or code.startswith("Q19") or code.startswith("Q20") \
       or code.startswith("Q21") or code.startswith("Q22") or code.startswith("Q23") \
       or code.startswith("Q24") or code.startswith("Q25") or code.startswith("Q26") \
       or code.startswith("Q27") or code.startswith("Q28") or code.startswith("Q29") \
       or code.startswith("Q30") or code.startswith("Q31") or code.startswith("Q32"):
        return "Ⅱ. 현재 체육활동 참여 현황"
    if code.startswith("Q7") or code.startswith("Q8") or code.startswith("Q9") \
       or code.startswith("Q10") or code.startswith("Q11") or code.startswith("Q12") \
       or code.startswith("Q13") or code.startswith("Q14"):
        return "Ⅲ. 체육활동 및 여건"
    if code.startswith("DQ"):
        return "Ⅴ. 개인 관련 사항"
    if code in DROP_VARS:
        return "파생변수·가중치"
    return "기타"

MAX_UNIQ = 50
rows = []
for i, code in enumerate(code_row):
    if pd.isna(code): continue
    code_str = str(code).strip()
    sec_str  = infer_section_2017(code_str)
    col_data = df[code_str] if code_str in df.columns else pd.Series(dtype=object)
    dtype    = "숫자" if pd.api.types.is_numeric_dtype(col_data) else "텍스트/혼합"
    uniq_vals = col_data.dropna().unique()
    try:    uniq_sorted = sorted(uniq_vals, key=lambda x: (isinstance(x, str), x))
    except: uniq_sorted = list(uniq_vals)
    n_uniq   = len(uniq_sorted)
    uniq_str = ", ".join(str(v) for v in uniq_sorted[:MAX_UNIQ])
    if n_uniq > MAX_UNIQ: uniq_str += "..."
    rows.append({
        "섹션":       sec_str,
        "변수코드":   code_str,
        "변수설명":   "",          # 2017은 설명행 없음
        "데이터타입": dtype,
        "유니크값수": n_uniq,
        "유니크값":   uniq_str,
        "표준컬럼명": "",
        "제거여부":   "Y" if code_str in DROP_VARS else "",
        "비고":       "",
    })

df_define = pd.DataFrame(rows)
print(f"\n변수정의서: {len(df_define)}개 변수")
print("\n섹션별 변수 수:")
print(df_define["섹션"].value_counts().to_string())

SEC_COLORS = {
    "관리 및 응답자 정보":         "D9E1F2",
    "Ⅰ. 건강과 체력상태":         "E2EFDA",
    "Ⅱ. 현재 체육활동 참여 현황": "FFF2CC",
    "Ⅲ. 체육활동 및 여건":        "FCE4D6",
    "Ⅴ. 개인 관련 사항":          "EDEDED",
    "파생변수·가중치":             "F4CCCC",
}
HDR_FILL  = PatternFill("solid", fgColor="1F4E79")
HDR_FONT  = Font(name="Arial", bold=True, color="FFFFFF", size=10)
BODY_FONT = Font(name="Arial", size=9)
thin      = Side(style="thin", color="CCCCCC")
BORDER    = Border(left=thin, right=thin, top=thin, bottom=thin)

df_define.to_excel(OUTPUT_FILE, index=False)
wb = load_workbook(OUTPUT_FILE)
ws = wb.active
ws.title = f"{YEAR}_variable_define"
ws.freeze_panes = "A2"

for cell in ws[1]:
    cell.fill = HDR_FILL; cell.font = HDR_FONT
    cell.alignment = Alignment(horizontal="center", vertical="center")
    cell.border = BORDER
ws.row_dimensions[1].height = 20

for r in range(2, ws.max_row + 1):
    sec_val  = str(ws.cell(r, 1).value or "")
    row_fill = PatternFill("solid", fgColor=SEC_COLORS.get(sec_val, "FFFFFF"))
    for c in range(1, ws.max_column + 1):
        cell = ws.cell(r, c)
        cell.fill = row_fill; cell.font = BODY_FONT
        cell.alignment = Alignment(vertical="center", wrap_text=(c in (3, 6)))
        cell.border = BORDER
    ws.row_dimensions[r].height = 30

col_widths = [24, 16, 50, 12, 10, 60, 16, 10, 20]
for i, w in enumerate(col_widths, 1):
    ws.column_dimensions[get_column_letter(i)].width = w

wb.save(OUTPUT_FILE)
print(f"\n✅ 저장 완료: {OUTPUT_FILE}")

총 컬럼수: 270
데이터: 9,000행 × 270열

변수정의서: 270개 변수

섹션별 변수 수:
섹션
Ⅱ. 현재 체육활동 참여 현황    177
Ⅲ. 체육활동 및 여건         62
Ⅰ. 건강과 체력상태          12
Ⅴ. 개인 관련 사항          11
관리 및 응답자 정보           7
파생변수·가중치              1

✅ 저장 완료: ..\data\3_variable_define\2017_variable_define.xlsx


# 2016

In [28]:
from pathlib import Path
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

YEAR        = 2016
RAW_2016    = Path(f"../data/1_raw/DATA_{YEAR}년 국민생활체육조사.xlsx")
CODEBOOK_F  = Path(f"../data/2_codebook/{YEAR}_codebook.xlsx")
OUTPUT_DIR  = Path("../data/3_variable_define")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = OUTPUT_DIR / f"{YEAR}_variable_define.xlsx"

# ══════════════════════════════════════════
# 1. 코드북 파싱 → {변수코드: 변수설명} 딕셔너리
# ══════════════════════════════════════════
df_cb = pd.read_excel(CODEBOOK_F, sheet_name="코드북")

# 변수명/설명 ffill (한 변수의 코드값이 여러 행에 걸쳐 있음)
df_cb["변수명_fill"]  = df_cb["변수명"].ffill()
df_cb["변수설명_fill"] = df_cb["변수설명"].ffill()
df_cb["구분_fill"]    = df_cb["구분"].ffill()

# 변수명이 처음 등장하는 행만 추출 → 변수별 설명 딕셔너리
var_rows  = df_cb.dropna(subset=["변수명"]).copy()
desc_dict = dict(zip(
    var_rows["변수명"].str.strip(),
    var_rows["변수설명"].str.strip()
))
sec_dict  = dict(zip(
    var_rows["변수명"].str.strip(),
    var_rows["구분_fill"].str.strip()
))

# 코드북 기반 섹션명 표준화
SEC_NORM = {
    "PART Ⅰ. 건강 및 체력상태":        "Ⅰ. 건강과 체력상태",
    "PART Ⅱ. 체육활동 참여 현황":       "Ⅱ. 현재 체육활동 참여 현황",
    "PART Ⅲ. 체육활동 및 여건":         "Ⅲ. 체육활동 및 여건",
    "PART Ⅴ. 개인 관련 사항":           "Ⅴ. 개인 관련 사항",
}
DROP_VARS = {"BA01","BA02","BA03","BA04","BA05","BA06","BA07","BA08","WT"}

def infer_section(code, sec_raw):
    if code in DROP_VARS: return "파생변수·가중치"
    for k, v in SEC_NORM.items():
        if isinstance(sec_raw, str) and k in sec_raw:
            return v
    # 코드패턴으로 보완
    code = str(code).strip()
    if code in ("ID","CITY","AREA","GENDER","SQ2","AGE","SUR"):
        return "관리 및 응답자 정보"
    if any(code.startswith(p) for p in ("Q1","Q2","Q3","Q4","Q5","Q6")):
        return "Ⅰ. 건강과 체력상태"
    if any(code.startswith(p) for p in ("Q15","Q16","Q17","Q18","Q19","Q20",
       "Q21","Q22","Q23","Q24","Q25","Q26","Q27","Q28","Q29","Q30","Q31","Q32","Q33")):
        return "Ⅱ. 현재 체육활동 참여 현황"
    if any(code.startswith(p) for p in ("Q7","Q8","Q9","Q10","Q11","Q12","Q13","Q14")):
        return "Ⅲ. 체육활동 및 여건"
    if code.startswith("DQ"):
        return "Ⅴ. 개인 관련 사항"
    return "기타"

print(f"코드북 변수 수: {len(desc_dict)}")

# ══════════════════════════════════════════
# 2. 원본 데이터 로드
# 2016 구조: row0=변수코드, row1~=데이터 (설명행 없음)
# ══════════════════════════════════════════
code_row = pd.read_excel(RAW_2016, header=None, nrows=1).iloc[0]
df = pd.read_excel(RAW_2016, header=0)
print(f"총 컬럼수: {len(code_row)}")
print(f"데이터: {df.shape[0]:,}행 × {df.shape[1]}열")

# ══════════════════════════════════════════
# 3. 변수정의서 생성
# ══════════════════════════════════════════
MAX_UNIQ = 50
rows = []

for i, code in enumerate(code_row):
    if pd.isna(code): continue
    code_str = str(code).strip()
    desc_str = desc_dict.get(code_str, "")
    sec_raw  = sec_dict.get(code_str, "")
    sec_str  = infer_section(code_str, sec_raw)

    col_data = df[code_str] if code_str in df.columns else pd.Series(dtype=object)
    dtype    = "숫자" if pd.api.types.is_numeric_dtype(col_data) else "텍스트/혼합"

    uniq_vals = col_data.dropna().unique()
    try:    uniq_sorted = sorted(uniq_vals, key=lambda x: (isinstance(x, str), x))
    except: uniq_sorted = list(uniq_vals)

    n_uniq   = len(uniq_sorted)
    uniq_str = ", ".join(str(v) for v in uniq_sorted[:MAX_UNIQ])
    if n_uniq > MAX_UNIQ: uniq_str += "..."

    rows.append({
        "섹션":       sec_str,
        "변수코드":   code_str,
        "변수설명":   desc_str,
        "데이터타입": dtype,
        "유니크값수": n_uniq,
        "유니크값":   uniq_str,
        "표준컬럼명": "",
        "제거여부":   "Y" if code_str in DROP_VARS else "",
        "비고":       "",
    })

df_define = pd.DataFrame(rows)
print(f"\n변수정의서: {len(df_define)}개 변수")
print(f"설명 채워진 변수: {(df_define['변수설명'] != '').sum()}개")
print("\n섹션별 변수 수:")
print(df_define["섹션"].value_counts().to_string())

# ══════════════════════════════════════════
# 4. Excel 저장
# ══════════════════════════════════════════
SEC_COLORS = {
    "관리 및 응답자 정보":         "D9E1F2",
    "Ⅰ. 건강과 체력상태":         "E2EFDA",
    "Ⅱ. 현재 체육활동 참여 현황": "FFF2CC",
    "Ⅲ. 체육활동 및 여건":        "FCE4D6",
    "Ⅴ. 개인 관련 사항":          "EDEDED",
    "파생변수·가중치":             "F4CCCC",
}
HDR_FILL  = PatternFill("solid", fgColor="1F4E79")
HDR_FONT  = Font(name="Arial", bold=True, color="FFFFFF", size=10)
BODY_FONT = Font(name="Arial", size=9)
thin      = Side(style="thin", color="CCCCCC")
BORDER    = Border(left=thin, right=thin, top=thin, bottom=thin)

df_define.to_excel(OUTPUT_FILE, index=False)
wb = load_workbook(OUTPUT_FILE)
ws = wb.active
ws.title = f"{YEAR}_variable_define"
ws.freeze_panes = "A2"

for cell in ws[1]:
    cell.fill = HDR_FILL; cell.font = HDR_FONT
    cell.alignment = Alignment(horizontal="center", vertical="center")
    cell.border = BORDER
ws.row_dimensions[1].height = 20

for r in range(2, ws.max_row + 1):
    sec_val  = str(ws.cell(r, 1).value or "")
    row_fill = PatternFill("solid", fgColor=SEC_COLORS.get(sec_val, "FFFFFF"))
    for c in range(1, ws.max_column + 1):
        cell = ws.cell(r, c)
        cell.fill = row_fill; cell.font = BODY_FONT
        cell.alignment = Alignment(vertical="center", wrap_text=(c in (3, 6)))
        cell.border = BORDER
    ws.row_dimensions[r].height = 30

col_widths = [24, 16, 50, 12, 10, 60, 16, 10, 20]
for i, w in enumerate(col_widths, 1):
    ws.column_dimensions[get_column_letter(i)].width = w

wb.save(OUTPUT_FILE)
print(f"\n✅ 저장 완료: {OUTPUT_FILE}")

코드북 변수 수: 285
총 컬럼수: 285
데이터: 9,012행 × 285열

변수정의서: 285개 변수
설명 채워진 변수: 285개

섹션별 변수 수:
섹션
Ⅰ. 건강과 체력상태     235
Ⅲ. 체육활동 및 여건     31
Ⅴ. 개인 관련 사항      10
파생변수·가중치          9

✅ 저장 완료: ..\data\3_variable_define\2016_variable_define.xlsx


In [29]:
from pathlib import Path
import pandas as pd
import numpy as np

YEAR      = 2016
MAPPING_F = Path(f"../data/4_mapping_table/{YEAR}_mapping.xlsx")
RAW_FILE  = Path(f"../data/1_raw/DATA_{YEAR}년 국민생활체육조사.xlsx")
CODEBOOK  = Path("../data/2_codebook/2025_codebook.xlsx")
OUT_DIR   = Path("../data/5_processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. 매핑 로드
df_var = pd.read_excel(MAPPING_F, sheet_name="변수_매핑", header=1)
df_val = pd.read_excel(MAPPING_F, sheet_name="값_매핑",   header=1)

rename_map, drop_cols = {}, set()
for _, row in df_var.iterrows():
    c_src = str(row.iloc[0]).strip()
    c_dst = str(row.iloc[3]).strip()
    if not c_src or c_src == "nan": continue
    if c_dst and c_dst != "nan": rename_map[c_src] = c_dst
    else: drop_cols.add(c_src)

val_rules = {}
for _, row in df_val.iterrows():
    유형  = str(row.iloc[0]).strip()
    c_src = str(row.iloc[2]).strip()
    c_dst = str(row.iloc[3]).strip()
    v_src = str(row.iloc[4]).strip()
    v_dst = str(row.iloc[5]).strip()
    if 유형 in ("분포차이(무시)","분포차이(무지)","보기 신규추가","동일"): continue
    if not c_src or c_src=="nan" or not c_dst or c_dst=="nan": continue
    if v_src in ("nan","(없음)","") or "..." in v_src: continue
    cur_col = rename_map.get(c_src, c_src)
    val_rules.setdefault(cur_col, [])
    if 유형 == "보기 삭제됨":
        for code in [v.strip() for v in v_src.split(",") if v.strip()]:
            val_rules[cur_col].append((code, np.nan, "삭제"))
    else:
        val_rules[cur_col].append((v_src, v_dst, "치환"))

print(f"리네임: {len(rename_map)}개 / 제거: {len(drop_cols)}개 / 값변환: {len(val_rules)}컬럼")

# 2. 코드북 로드
df_cb = pd.read_excel(CODEBOOK, sheet_name="코드정의")
code_map = {}
for _, row in df_cb.iterrows():
    col       = str(row["변수코드"]).strip()
    label_val = str(row.get("코드레이블","")).strip()
    code_val  = row["코드값"]
    if not label_val or label_val in ("nan",""): continue
    if str(code_val).strip() in ("(연속형/자유값)","(주관식)","(없음)","nan"): continue
    code_map.setdefault(col, {})
    try:    code_map[col][int(float(code_val))] = label_val
    except: pass
    code_map[col][str(code_val).strip()] = label_val
print(f"코드북: {len(code_map)}개 컬럼")

# 3. 원본 로드
# 2016 구조: row0=변수코드, row1~=데이터 (설명행 없음)
df = pd.read_excel(RAW_FILE, header=0)
print(f"원본: {df.shape[0]:,}행 × {df.shape[1]}열")
print(f"컬럼 확인 (앞 6개): {list(df.columns[:6])}")  # ID CITY AREA GENDER SQ2 AGE 확인

# 4. 컬럼 제거 & 리네임
df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")
df = df.rename(columns=rename_map)
df = df.loc[:, ~df.columns.duplicated()]
print(f"리네임 후: {df.shape[1]}열")

# 5. 연도간 값 변환 (AREA→시도명)
changed_diff = 0
for col, rules in val_rules.items():
    if col not in df.columns: continue
    has_text = any(isinstance(v,str) and v not in ("nan","") for _,v,_ in rules
                   if not (isinstance(v,float) and np.isnan(v)))
    if has_text and pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].astype(object)
    for v_src, v_dst, _ in rules:
        mask = df[col].astype(str).str.strip() == str(v_src).strip()
        try:
            nv = int(v_src) if str(v_src).isdigit() else float(v_src)
            mask = mask | (df[col] == nv)
        except: pass
        cnt = int(mask.sum())
        if cnt > 0:
            df.loc[mask, col] = v_dst
            changed_diff += cnt
print(f"연도간 값 변환: {changed_diff:,}건")

# 6. 코드북 레이블 치환
changed_label = 0
for col, mapping in code_map.items():
    if col not in df.columns: continue
    if pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].astype(object)
    before = df[col].copy()
    df[col] = df[col].map(lambda x: mapping.get(x, mapping.get(str(x).strip(), x)))
    changed_label += (before.astype(str) != df[col].astype(str)).sum()
print(f"코드북 레이블 치환: {changed_label:,}건")

# 7. 2025 컬럼 순서 맞춤
ref = Path("../data/5_processed/survey_2025_standardized.csv")
if ref.exists():
    cols_ref = [c for c in pd.read_csv(ref, nrows=0).columns if c != "연도"]
    ordered  = [c for c in cols_ref if c in df.columns]
    extra    = [c for c in df.columns if c not in cols_ref]
    df = df[ordered + extra]
    if extra: print(f"⚠️  {YEAR}에만 있는 컬럼 ({len(extra)}개): {extra}")

# 8. 연도 추가 & 저장
df.insert(0, "연도", YEAR)
out_csv = OUT_DIR / f"survey_{YEAR}_standardized.csv"
df.to_csv(out_csv, index=False, encoding="utf-8-sig")
print(f"\n✅ [{df.shape[0]:,}행 × {df.shape[1]}열] 저장 완료: {out_csv}")

리네임: 150개 / 제거: 135개 / 값변환: 16컬럼
코드북: 6개 컬럼
원본: 9,012행 × 285열
컬럼 확인 (앞 6개): ['ID', 'CITY', 'AREA', 'GENDER', 'SQ2', 'AGE']
리네임 후: 150열
연도간 값 변환: 0건
코드북 레이블 치환: 0건

✅ [9,012행 × 151열] 저장 완료: ..\data\5_processed\survey_2016_standardized.csv


# Merge

In [8]:
from pathlib import Path
import pandas as pd

PROCESSED_DIR = Path("../data/5_processed")
OUTPUT_FILE   = Path("../data/sports_survey.csv")

# 1. 표준화 파일 전체 로드
csv_files = sorted(PROCESSED_DIR.glob("survey_*_standardized.csv"))
print(f"발견된 파일 ({len(csv_files)}개):")

dfs = []
for f in csv_files:
    df = pd.read_csv(f, encoding="utf-8-sig", low_memory=False)
    year = df["연도"].iloc[0] if "연도" in df.columns else "unknown"
    print(f"  {f.name}: {df.shape[0]:,}행 × {df.shape[1]}열  (연도={year})")
    dfs.append(df)

# 2. 통합
df_merged = pd.concat(dfs, ignore_index=True, sort=False).copy()
cols_order = ["연도"] + [c for c in df_merged.columns if c != "연도"]
df_merged  = df_merged[cols_order]

print(f"\n[통합 결과] {df_merged.shape[0]:,}행 × {df_merged.shape[1]}열")
print("\n연도별 행수:")
print(df_merged["연도"].value_counts().sort_index().to_string())

# 3. 저장
df_merged.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
print(f"\n✅ 저장 완료: {OUTPUT_FILE}")

발견된 파일 (2개):
  survey_2024_standardized.csv: 9,000행 × 213열  (연도=2024)
  survey_2025_standardized.csv: 9,000행 × 229열  (연도=2025)

[통합 결과] 18,000행 × 229열

연도별 행수:
연도
2024    9000
2025    9000

✅ 저장 완료: ..\data\sports_survey.csv
